# Array min by repeated kernel launches

A reduction can also skip atomics and grid-stride loops altogether. Each launch shrinks the array by a large factor, and the host keeps launching until one value is left:

```
n = 5000  ->  ceil(5000 / 128) = 40 partials  ->  1 partial  ->  done
```

Write `array_min(x)` that returns the smallest element of a 1-D `float32` array this way.

## The kernel: `min_pass(src, dst)`
- Blocks of **64 threads**, and each block covers **128** consecutive elements of `src`: block `b` owns indices `[128 b, 128 b + 128)`.
- **First step during the load**: thread `t` reads elements `base + t` and `base + t + 64` (where they exist) and stores their min in shared memory. This halves the number of threads that sit idle in the first tree step.
- A tree reduction over the 64 shared values; thread 0 writes the block's min to `dst[blockIdx.x]`.
- Indices past the end of `src` contribute $+\infty$, the identity of min.

## The host: `array_min(x)`
- Launch `min_pass` with $\lceil n/128 \rceil$ blocks, feed its output into the next launch, and stop when one element remains. Keep the intermediate arrays on the device: no copies to the host between passes.
- `x`: `float32`, shape `(n,)` with `n >= 1`, no NaNs. Return the minimum as a Python `float`. For `n == 1`, no launch is needed.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32

TPB = 64            # threads per block
PER_BLOCK = 2 * TPB  # elements per block

def array_min(x):
    """x: float32 (n,), n >= 1. Returns min(x) as a Python float."""
    d = cuda.to_device(x)
    blocks = (x.shape[0] + PER_BLOCK - 1) // PER_BLOCK
    out = cuda.device_array(blocks, dtype=np.float32)
    min_pass[blocks, TPB](d, out)
    return float(out.copy_to_host()[0])   # TODO: only one pass; block 0's min is not the global min

@cuda.jit
def min_pass(src, dst):
    pass  # TODO


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [4.0, 1.5, 9.0, -2.0, 3.0], "dtype": "float32"}], "e": -2.0}, {"a": [{"__nd__": [7.0], "dtype": "float32"}], "e": 7.0}, {"a": [{"__nd__": [128.0, 129.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 137.0, 138.0, 139.0, 140.0, 141.0, 142.0, 143.0, 144.0, 145.0, 146.0, 147.0, 148.0, 149.0, 150.0, 151.0, 152.0, 153.0, 154.0, 155.0, 156.0, 157.0, 158.0, 159.0, 160.0, 161.0, 162.0, 163.0, 164.0, 165.0, 166.0, 167.0, 168.0, 169.0, 170.0, 171.0, 172.0, 173.0, 174.0, 175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188.0, 189.0, 190.0, 191.0, 192.0, 193.0, 194.0, 195.0, 196.0, 197.0, 198.0, 199.0, 200.0, 201.0, 202.0, 203.0, 204.0, 205.0, 206.0, 207.0, 208.0, 209.0, 210.0, 211.0, 212.0, 213.0, 214.0, 215.0, 216.0, 217.0, 218.0, 219.0, 220.0, 221.0, 222.0, 223.0, 224.0, 225.0, 226.0, 227.0, 228.0, 229.0, 230.0, 231.0, 232.0, 233.0, 234.0, 235.0, 236.0, 237.0, 238.0, 239.0, 240.0, 241.0, 242.0, 243.0, 244.0, 245.0, 246.0, 247.0, 248.0, 249.0, 250.0, 251.0, 252.0, 253.0, 254.0, 255.0], "dtype": "float32"}], "e": 128.0}, {"a": [{"__nd__": [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0], "dtype": "float32"}], "e": 5.0}, {"a": [{"__nd__": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0], "dtype": "float32"}], "e": -1.0}, {"a": [{"__nd__": [{"__f__": "inf"}, 3.0, {"__f__": "-inf"}, 0.0], "dtype": "float32"}], "e": {"__f__": "-inf"}}, {"a": [{"__nd__": [0.9091452360153198, 0.751396656036377, -0.025538381189107895, 0.65250563621521, -0.6509536504745483, -1.829492449760437, 1.6785105466842651, 0.47661399841308594, -2.066348075866699, -1.0589356422424316, -0.6195720434188843, 0.341558575630188, 1.326922059059143, 0.30765286087989807, -0.4329802393913269, 0.4673631489276886, -0.2221706360578537, 0.18286697566509247, -0.6662219762802124, -1.206809401512146, 1.283158540725708, -0.31994926929473877, 0.1182897537946701, -0.600803792476654, -0.5448718667030334, -0.3537006080150604, 0.6579650640487671, 1.2185423374176025, 0.5426804423332214, -0.5538851618766785, 2.796293020248413, 0.8393442630767822, 0.9432998895645142, -0.5472232699394226, -0.24332360923290253, 0.6090924143791199, -1.739393949508667, -0.7746402025222778, 0.39951542019844055, 2.4215543270111084, 0.42384257912635803, -1.0369771718978882, -1.2175779342651367, 0.9792370796203613, -0.8115336894989014, -0.3445526659488678, 1.9637389183044434, 0.060207098722457886, -0.35659685730934143, -0.7827652096748352, 1.3230475187301636, 1.3185503482818604, -0.6749138832092285, -1.41538405418396, -0.12107568234205246, 1.3762589693069458, 0.45615628361701965, -1.2143839597702026, 0.3349961042404175, 0.7684516906738281, -0.030736522749066353, 0.1655428260564804, 0.0014840556541457772, -0.5039287209510803, -0.6634593605995178, -1.3689870834350586, -1.196999430656433, -1.7379549741744995, 0.4364594519138336, 0.4774486720561981, 0.05046604201197624, 0.42421454191207886, 0.04268765076994896, -0.7787262797355652, 0.16877831518650055, -0.3481200635433197, -0.2337730973958969, -2.1181840896606445, -0.567257821559906, 0.38518816232681274, -0.38857412338256836, -0.8917727470397949, -1.6704230308532715, -0.7189392447471619, 1.3713593482971191, -0.6672164797782898, 1.0095701217651367, -0.5217648148536682, 1.0145933628082275, -0.5757256746292114, 0.3449898660182953, 0.61173015832901, 0.5815532207489014, -0.9718542695045471, -0.36845842003822327, -0.8926870226860046, -0.15297284722328186, -0.3429906666278839, -0.26835018396377563, 0.5973858833312988, -1.0584485530853271, -0.8106234073638916, -0.5956935286521912, -0.25307196378707886, -0.8134654760360718, 0.5010349750518799, -1.09351646900177, 1.7625378370285034, 0.04727428779006004, 1.176220178604126, -1.0800106525421143, -0.31013667583465576, 0.1391427218914032, 1.7808692455291748, 0.12659990787506104, 1.5109045505523682, -0.6808055639266968, 1.5353447198867798, -0.6463974714279175, -1.7464724779129028, -1.9432018995285034, 0.23347431421279907, -0.18872399628162384, 0.27403247356414795, -0.04545208439230919, 0.5153059363365173, 1.6554505825042725, -0.17505472898483276, 0.3076832890510559, -0.35897618532180786, -1.6976279020309448, -0.24440769851207733, 1.538963794708252, -0.7775423526763916, 2.346444845199585, 0.6095312237739563, 0.5105674266815186, 0.9954822659492493, 0.07455846667289734, 0.10582949221134186, 1.3993648290634155, -0.6544346809387207, -0.0759725272655487, -1.1163722276687622, 0.18129922449588776, 3.1227142810821533, 1.2630432844161987, -0.8636914491653442, -1.694982647895813, 0.6340669989585876, 0.7768086791038513, -0.5340918898582458, 1.414341926574707, 0.5010877251625061, 0.7997105717658997, 0.8513801693916321, 1.1934449672698975, -0.35108932852745056, -0.6055095791816711, 1.4581940174102783, -0.04919041320681572, -1.1245629787445068, 0.46200481057167053, -0.3694256544113159, 0.42201027274131775, 3.270291805267334, 1.8933122158050537, -0.338782399892807, -1.1198426485061646, 0.10068386048078537, -0.847409188747406, 0.3456473648548126, 0.4024438261985779, 0.7683627605438232, 0.11129456013441086, -1.2706085443496704, -0.029761331155896187, 0.17887073755264282, 0.33945971727371216, 1.315692663192749, -0.9244818091392517, -0.047430846840143204, -0.48976844549179077, -0.7373855113983154, 0.732300877571106, -0.5270559787750244, 2.0237109661102295, -2.4157650470733643, 1.5817680358886719, -0.1578385829925537, -1.1555458307266235, 0.2468298077583313, 0.19598235189914703, 1.0455293655395508, 1.3199856281280518, -0.6983813047409058, 0.06962395459413528, 0.40598264336586, 0.26085183024406433, -0.2752487361431122, 1.420381784439087, 0.3629491329193115, -1.262918472290039, 0.3405264616012573, -0.18930405378341675, -0.2287617176771164, 0.89976966381073, 0.7091174721717834, 0.3496178984642029, 0.4838322401046753, 0.9349967837333679, -0.5542901158332825, 0.1566918045282364, 1.3204736709594727, -0.5404009819030762, -0.4116772711277008, -0.5710806250572205, 1.0947208404541016, -0.4357708990573883, 1.113878607749939, -0.37649449706077576, 0.23457588255405426, 0.34742864966392517, -0.49569129943847656, -0.9709481596946716, 0.9900400042533875, -0.3697275221347809, 0.5393691062927246, 0.6814695596694946, -1.4221874475479126, 1.2543219327926636, 0.1513684093952179, -0.28400957584381104, -0.3543475270271301, -0.4627762734889984, 0.27104395627975464, 2.106262445449829, -0.07819739729166031, -1.0869605541229248, -0.2296585887670517, 2.121279239654541, 0.1135462298989296, 1.9812371730804443, -0.725770115852356, -0.7405781745910645, -1.1151918172836304, -0.9181954264640808, -0.039426617324352264, 0.1413169801235199, -1.7444369792938232, -1.026956558227539, -0.7957781553268433, -0.39579111337661743, 1.348694920539856, 1.4942755699157715, 0.4031030535697937, 0.7275587320327759, -0.8743171691894531, 0.27594512701034546, -1.8224960565567017, -0.1421888768672943, 0.20548008382320404, 0.7457151412963867, -0.2286297082901001, -0.2279970794916153, -1.1240564584732056, -2.3853747844696045, -0.7836164236068726, -2.068225860595703, -0.5506588220596313, -0.4706538915634155, -1.5595333576202393, -1.9951859712600708, -0.35202452540397644, -0.7631328105926514, 0.2619978189468384, 0.7144916653633118, 1.4519906044006348, 0.8120233416557312, -1.07481849193573, -0.2209768146276474, 1.954412579536438, -1.017076015472412, -0.5029299259185791, 1.1852385997772217, 0.46087589859962463, -0.7526391744613647, 0.2153128683567047, -0.7287991046905518, 0.7729450464248657, -0.24055573344230652, -0.06911807507276535, 0.08584049344062805, -2.0993363857269287, -0.2681213617324829, 0.3025299310684204, -0.026171326637268066, -0.6248458027839661, -0.1652376353740692, 1.3855087757110596, 0.5977359414100647, -0.7083663940429688, 1.0058048963546753, 0.23022441565990448, -0.5575807094573975, -1.8483197689056396, 0.5161685347557068, -1.1805614233016968, 0.2888573706150055, 1.0544242858886719, 0.6424434185028076, -1.2576680183410645, -0.9708150625228882, 0.6207188367843628, 1.0506975650787354, -0.02780618704855442, -0.18456019461154938, 1.8017358779907227, 1.6471821069717407, 0.2477646768093109, -0.3709065318107605, 0.6542377471923828, -1.2326922416687012, 0.21543295681476593, 0.25527116656303406, 1.2420190572738647, 0.5690380930900574, -1.8022809028625488, 1.8824193477630615, 0.605972409248352, -1.9884860515594482, -1.5339481830596924, 0.7141531109809875, -1.2994778156280518, 0.8351976871490479, -1.0448249578475952, -1.5839836597442627, 1.6116702556610107, 1.7583860158920288, -0.26755860447883606, 1.6597031354904175, 0.3909919857978821, 0.5468053817749023, -0.14253565669059753, 0.728629469871521, 1.5519143342971802, 1.578812837600708, 2.2014431953430176, 1.9622002840042114, 0.7996727824211121, -0.35595110058784485, 1.024437427520752, 0.8276326656341553, -0.23709779977798462, -0.9754621982574463, -0.638407826423645, -1.3712859153747559, -0.7137768864631653, 0.3732614815235138, 1.4474247694015503, 0.3035663068294525, -0.3519638180732727, -0.2778376638889313, 0.019470414146780968, 1.4242156744003296, 0.30008769035339355, -1.6232192516326904, 0.16236907243728638, -1.5929030179977417, -0.5718655586242676, 0.23102831840515137, -0.7597147226333618, -0.013050349429249763, 1.3083237409591675, -0.7139818072319031, 1.1701867580413818, 0.9908804893493652, 0.7936769127845764, -0.7621508836746216, 1.1996322870254517, 0.4950481355190277, 0.8450272083282471, 0.1512802094221115, 1.3877304792404175, -1.2281311750411987, 0.5221574902534485, 0.9111799597740173, 2.883098602294922, -1.44236421585083, -1.0733152627944946, -0.111595019698143, 0.376595139503479, -0.7026156783103943, 0.8138883113861084, -0.1470351219177246, 0.17742717266082764, -1.2815169095993042, 0.4672873914241791, 0.15697509050369263, 1.5343655347824097, 0.836412250995636, 1.5289503335952759, 0.3739042282104492, -0.7091265320777893, 0.7337813377380371, -1.4827473163604736, -1.367656946182251, -0.6177063584327698, 0.8262739181518555, 1.0577003955841064, -1.7364944219589233, -0.347487211227417, 1.1677963733673096, -0.446077823638916, -1.6667827367782593, 0.016953008249402046, 1.3202569484710693, -0.6402636170387268, -0.5978428721427917, 2.0237748622894287, 0.6930892467498779, -1.126341462135315, 0.8435807228088379, 1.2407896518707275, -0.2895684540271759, 0.23124070465564728, 0.8950998187065125, 0.36733394861221313, 1.60242760181427, 1.40864098072052, 1.0186609029769897, 1.6473314762115479, -0.20468342304229736, -0.698321521282196, -1.2553324699401855, -0.46552082896232605, -1.0098094940185547, -0.1282464861869812, 1.0941838026046753, 1.6391726732254028, 0.8438020348548889, 0.7607465386390686, 0.4712028205394745, 0.7049767374992371, -0.3210465908050537, -0.45180216431617737, -1.0270161628723145, -0.1374565213918686, 0.5068448185920715, 0.02087324671447277, 0.3549448549747467, 2.0584700107574463, 0.6086518168449402, 0.7495012283325195, 0.18689890205860138, 0.7069418430328369, -0.8654841780662537, 0.7071611285209656, 0.831053614616394, 1.041282296180725, -1.3595409393310547, -0.11544342339038849, 1.4165005683898926, 0.8309277296066284, -1.3015881776809692, -0.7743418216705322, -1.0717357397079468, 0.43998071551322937, 0.9014857411384583, -0.23115035891532898, -1.6513793468475342, 0.15680311620235443, -1.9894167184829712, 0.9731670022010803, -0.970041036605835, -1.0382031202316284, -1.1690106391906738, -0.09285133332014084, 0.013536207377910614, 0.3444458246231079, -0.198101207613945, -0.17382580041885376, 0.9643125534057617, 1.3058661222457886, 0.2334502637386322, 0.5488958358764648, 0.8025863766670227, -0.18039719760417938, -0.5748371481895447, -1.3148648738861084, -0.44988030195236206, 1.6296535730361938, -0.6537941694259644, -1.2137738466262817, -0.16717006266117096, 0.2884588837623596, 0.42151129245758057, -0.8508881330490112, 0.4673983156681061, -0.4234365224838257, -0.3558313250541687, -0.17420192062854767, -0.8568778038024902, -0.5128885507583618, 0.7280361652374268, -1.014059066772461, -0.32004478573799133, 0.5309084057807922, 0.26504388451576233, 0.28004190325737, -1.1117174625396729, -1.2276365756988525, -0.20109406113624573, -0.5638948678970337, -1.727338194847107, -0.04978738725185394, 0.4093228876590729, 1.430583119392395, -0.09605419635772705, -0.17857517302036285, -0.7523543238639832, 0.24610763788223267, -0.27302828431129456, 2.0085933208465576, -0.6715664267539978, 1.3069305419921875, -0.14502641558647156, -1.0891458988189697, 0.45765137672424316, 0.37982121109962463, 1.5731074810028076, 0.8131134510040283, 0.13608333468437195, -0.33556148409843445, -0.5811773538589478, -0.009297695942223072, 0.3009214699268341, -0.7045159935951233, -0.6689944267272949, 0.4192639887332916, -0.8097174763679504, -0.043839231133461, -0.20868702232837677, -0.6491132974624634, 1.3833976984024048, 1.0367720127105713, 0.09347397834062576, -1.1493033170700073, 0.32050907611846924, -0.07156786322593689, -0.0032225048635154963, 0.09839580953121185, -0.4483513832092285, 2.3140945434570312, 0.6911028027534485, -0.09214077889919281, 0.39233019948005676, -0.33416563272476196, -0.6671651601791382, -0.7708912491798401, 0.4642789363861084, 1.2298011779785156, 1.4919792413711548, 0.384874165058136, 0.43441715836524963, -1.529665231704712, -1.7712604999542236, -0.2859216332435608, 0.9777535200119019, 1.380761981010437, -0.10205692797899246, -1.104822039604187, -0.18955731391906738, -0.005373646505177021, -0.874709963798523, -0.18279078602790833, 0.9054862856864929, -0.5343042016029358, -0.05631136894226074, 1.853711724281311, 0.5008032917976379, 0.5248970985412598, -0.4324527978897095, 0.8990617394447327, -1.426055669784546, 1.343170404434204, 0.02770661562681198, -0.8153409361839294, -0.6277049779891968, -0.17243856191635132, -2.2824721336364746, -0.3836568295955658, -0.13104355335235596, -1.494796872138977, 0.11324339359998703, 0.8678916096687317, -1.7272818088531494, -0.9649848341941833, 0.35911473631858826, 1.8627071380615234, -0.6017400622367859, 0.08634036779403687, -0.7403022050857544, 0.055068157613277435, 0.47381529211997986, -0.58563232421875, 0.6268421411514282, 3.192225933074951, 1.5886867046356201, -0.057998176664114, 0.2106437385082245, -1.3942116498947144, -0.28268516063690186, -0.5656148791313171, -0.9954285025596619, -2.0706846714019775, 1.358324408531189, 0.3267059922218323, -1.3083094358444214, 0.5868219137191772, 0.5822248458862305, -1.4160654544830322, 1.1056134700775146, -2.386249303817749, 0.6972182393074036, -1.1692084074020386, 0.22371980547904968, -1.9941010475158691, 0.0042913202196359634, 0.9470945596694946, 1.0844465494155884, -0.02788473106920719, 0.044305622577667236, -0.916591465473175, -0.610107421875, -0.15272316336631775, -1.4462579488754272, 0.160427525639534, -0.4803430140018463, 0.23940441012382507, 0.62237548828125, -0.3632601201534271, -1.153975486755371, 0.27643609046936035, -0.7427724599838257, -0.1718233972787857, -0.046720951795578, -1.2581673860549927, 0.5468945503234863, -1.2545117139816284, -0.14664222300052643, 0.7448040843009949, 0.5238490104675293, 0.31960394978523254, 1.7674541473388672, 0.05276912450790405, 0.36281946301460266, 1.1533689498901367, -1.4953941106796265, -0.21769767999649048, 0.9909554123878479, 0.6506264209747314, -0.44256871938705444, 0.6599729657173157, 0.7298528552055359, 0.19010084867477417, 1.2723544836044312, -1.0262501239776611, 0.7585944533348083, -0.030516229569911957, -0.13403776288032532, -1.7572300434112549, 2.326103925704956, -2.473768711090088, 0.13299179077148438, -0.18118378520011902, 1.8136827945709229, -0.564948320388794, -0.20435920357704163, 0.18585506081581116, 0.7342908382415771, 1.1163750886917114, 0.6588415503501892, -2.119706869125366, -0.6126701235771179, 0.7131288051605225, 0.6850065588951111, 1.1311372518539429, -0.5610557198524475, -0.13516463339328766, 0.2078658640384674, 0.9475046396255493, 1.289055347442627, 0.10411754995584488, 1.4103440046310425, 0.29942095279693604, 0.7234560251235962, -1.2884725332260132, -0.9567473530769348, 1.3740642070770264, -0.4177803099155426, 1.6244722604751587, 1.2292653322219849, -1.255518913269043, -0.10253490507602692, 0.21094951033592224, -0.37096869945526123, -2.156303882598877, 0.25768306851387024, -0.21254310011863708, -1.4667384624481201, -0.4262458086013794, -0.5631744861602783, 1.9780791997909546, 0.3135257661342621, -0.940037190914154, -0.18069317936897278, -1.9780153036117554, 0.5576352477073669, -0.15358230471611023, 1.1484287977218628, -0.08838891237974167, 1.3895387649536133, 1.4458187818527222, -1.17701256275177, -0.30234625935554504, 0.10123076289892197, -0.08327844738960266, -0.8221677541732788, -0.16571015119552612, -0.12097221612930298, 0.8321853280067444, -0.1995605230331421, 0.2033035159111023, 1.2753162384033203, 0.5835735201835632, -0.4486251175403595, -1.0381555557250977, 0.15126241743564606, -0.9515186548233032, 0.5427514314651489, -0.35200434923171997, -1.669421672821045, 0.6212775707244873, -0.9361976981163025, 0.47176268696784973, 0.10355309396982193, 0.37640056014060974, -1.87371027469635, -0.6362054347991943, -0.2530844509601593, 2.4653329849243164, 0.8722749948501587, 0.430868923664093, 1.3707069158554077, -0.015600431710481644, -0.0018052698578685522, -0.695354700088501, 0.3029091954231262, -2.227473020553589, -1.7713096141815186, 1.0537174940109253, 1.0018012523651123, 0.2252741903066635, -0.35841846466064453, 1.5824940204620361, 0.9763016104698181, -1.0770660638809204, 0.30917268991470337, 0.8582093119621277, -0.36190080642700195, -1.8398226499557495, -0.26579537987709045, 0.47477516531944275, 0.6954581141471863, -0.5537970662117004, 0.3559601306915283, -0.01778818853199482, 0.6153730750083923, -1.1753029823303223, -0.7010002732276917, 0.5823771357536316, 1.0276669263839722, 0.10478820651769638, 0.6929087042808533, 0.3541766405105591, -0.6961858868598938, 1.6315363645553589, -0.7714423537254333, -0.9512416124343872, 0.15837544202804565, 0.8309364318847656, 0.35353517532348633, -0.7464908361434937, -1.3096810579299927, 0.4659672975540161, 0.2028207629919052, -0.906030535697937, -0.5704389214515686, 1.2706876993179321, 0.8747434020042419, -2.430255651473999, -0.6717202067375183, 1.251359224319458, -0.5584206581115723, -2.103818655014038, -1.1727323532104492, 1.0367283821105957, -0.9713312387466431, 0.6526517271995544, -1.2183820009231567, 0.8590167760848999, -0.37396231293678284, 0.21047373116016388, -0.014626611955463886, -0.5434115529060364, -2.784036874771118, 0.8537082672119141, 0.8340593576431274, 0.5499813556671143, 0.9164892435073853, -0.7431626319885254, 0.4655832052230835, -1.6707563400268555, -1.7099806070327759, -0.016048936173319817, -2.162982940673828, 0.575541615486145, -0.30731356143951416, -0.04925854131579399, -0.9056388139724731, 0.1845501810312271, 0.6083475947380066, -1.403961181640625, -0.9479118585586548, 0.39367884397506714, -1.4289089441299438, 0.44903552532196045, -1.1253210306167603, -0.5637479424476624, 0.43871554732322693, 0.5005602836608887, -1.261154055595398, 1.5682011842727661, -0.5253064632415771, -0.8254006505012512, 0.6317103505134583, 0.02524285949766636, 1.8106160163879395, -0.7396266460418701, -1.8826905488967896, -2.4564871788024902, -0.7181307673454285, 0.011972314678132534, 0.4622918367385864, 0.5772647261619568, 1.1029505729675293, -0.31629064679145813, -0.7671296000480652, 2.975752115249634, 2.490368366241455, -0.42020025849342346, 0.22475457191467285, -0.2668314278125763, 1.948543906211853, 1.303618311882019, -2.0922727584838867, 0.09705890715122223, -0.5814762115478516, 0.3192282021045685, -0.566024661064148, 0.16347430646419525, -0.8265029788017273, 1.0317658185958862, -0.45011115074157715, 1.1109213829040527, -1.2918288707733154, 0.5578044652938843, 0.6422168016433716, 0.27054810523986816, -0.2432187795639038, -1.0671039819717407, -0.5169020891189575, -1.0869675874710083, 0.890397310256958, 0.3740832507610321, 0.6521720886230469, 0.9106080532073975, -1.1024454832077026, -0.5972152352333069, -0.5383408665657043, -0.8290457129478455, -0.9631597995758057, -1.2163251638412476, -0.47672027349472046, 2.089791774749756, 0.3857766091823578, 0.35347864031791687, 0.2714163661003113, -1.2748346328735352, 0.23460081219673157, -0.3272554874420166, -0.016450054943561554, -0.7178349494934082, 1.7617806196212769, -1.7184773683547974, -0.5063900947570801, -0.880744993686676, -0.6138949394226074, 1.2232567071914673, 1.4223307371139526, 0.32840579748153687, -1.2782129049301147, 0.753532886505127, -0.25154829025268555, -2.1567647457122803, 0.35635295510292053, -0.46514177322387695, -0.2554011940956116, 0.5845399498939514, 1.2532819509506226, -1.4089045524597168, 0.07693247497081757, 0.20397642254829407, 0.07777556777000427, -0.21765176951885223, 1.3147121667861938, -0.9213704466819763, -0.3336472809314728, 1.7591582536697388, -0.039182037115097046, 0.8149217963218689, 1.8750743865966797, -0.7344515919685364, -1.659938931465149, -2.90478777885437, -0.19800560176372528, -0.09571994841098785, -0.6341720819473267, 0.5937468409538269, 1.3848110437393188, 0.3811989724636078, -0.5964627861976624, -1.6784098148345947, -0.14533498883247375, 0.15226373076438904, 1.1469464302062988, -1.293699026107788, -1.1642040014266968, -2.498758316040039, -0.8018798232078552, 0.8009907603263855, 1.0780445337295532, -0.22736856341362, -1.2719664573669434, 0.25804686546325684, -1.1167951822280884, -0.9105892181396484, 2.4417827129364014, -1.066670298576355, -0.1138966903090477, -0.3352922201156616, 0.6197181344032288, -1.046057939529419, -0.5548611283302307, 1.007077932357788, -0.8977115154266357, -0.9843265414237976, -1.0072575807571411, -1.8071069717407227, -2.2587993144989014, 0.2836449146270752, -1.4503849744796753, -0.9613925814628601, 0.7919269800186157, 0.19512102007865906, -1.3539338111877441, 0.5359559655189514, -0.14897064864635468, -0.8627922534942627, 1.861272931098938, -0.6143613457679749, -0.44648870825767517, 0.19788971543312073, 0.10462010651826859, 0.08089981973171234, -0.4125954806804657, -0.3781803548336029, 0.31878000497817993, -0.5344683527946472, -0.24528764188289642, 0.740532398223877, 0.705873966217041, -0.3021968901157379, -0.6717990040779114, 1.5205687284469604, -1.6167007684707642, 0.2483087182044983, 0.2583657205104828, -1.573701024055481, -1.3680397272109985, 1.4957447052001953, 0.5458137392997742, -0.4052715599536896, -0.15770451724529266, 0.2909759283065796, -1.4481056928634644, 0.9458619952201843, -2.121086835861206, 0.044000472873449326, -0.8091015219688416, -0.9081604480743408, 0.3472578823566437, -0.7573161125183105, 1.251865267753601, 1.2769591808319092, 1.5375186204910278, 0.6298075914382935, -0.8387480974197388, 0.6204577088356018, -1.8439176082611084, 2.141561508178711, 0.8096054792404175, 0.40460604429244995, -2.2536258697509766, 1.1195186376571655, -0.23229250311851501, -0.08647549152374268, -1.0271337032318115, -3.0192437171936035, -0.5152630805969238, 0.38647693395614624, -0.6277574300765991, 0.81410151720047, 1.1894612312316895, -0.2946908175945282, 0.9478722214698792, -0.7785608172416687, -0.6921042203903198, -0.8096461296081543, -0.3902507424354553, 1.1577824354171753, -0.32821494340896606, -0.7920989394187927, 0.17290282249450684, -1.5683544874191284, -1.1254159212112427, 0.919845461845398, 1.3887090682983398, 1.4664093255996704, -0.939342737197876, 0.0831054076552391, 0.10059799253940582, 1.0846720933914185, 0.6835044622421265, -0.5549231171607971, -1.603941798210144, 0.007710332982242107, -1.1192909479141235, 1.8730180263519287, 0.46029040217399597, 2.2213921546936035, 1.087730884552002, -0.4498593509197235, -0.6384304761886597, -2.3002984523773193, 0.5228608846664429, 2.148369550704956, -0.4567943215370178, -0.07252765446901321, -0.873583197593689, -0.6369485855102539, 0.8357915878295898, 1.3046536445617676, -0.8370171189308167, 0.1841002106666565, -2.5134828090667725, -0.8175836801528931, -0.2330353558063507, 1.2996546030044556, -0.167134627699852, 2.0160956382751465, -0.07985066622495651, -0.16778147220611572, -0.16727463901042938, -0.10607623308897018, -1.7044034004211426, -0.16320817172527313, 0.3218328356742859, 1.3007838726043701, 0.6471633911132812, -0.3630144000053406, 1.1265558004379272, -1.8275450468063354, -0.32923537492752075, 0.7053645253181458, 0.059159595519304276, 0.11244015395641327, -1.017560362815857, -1.5449576377868652, -0.021026385948061943, 1.0965275764465332, -0.46238526701927185, 0.21812868118286133, 0.830761730670929, -1.2919543981552124, 0.8544542789459229, -0.09741132706403732, -0.9957367181777954, 0.38517850637435913, 0.3692152202129364, -0.9463017582893372, 0.18042109906673431, -0.9057701826095581, 0.5176974534988403, -0.8381655216217041, -0.677290678024292, -0.34112757444381714, -1.3370046615600586, 0.1790929138660431, -0.6017690300941467, 1.374635934829712, 0.06968408823013306, -0.25707733631134033, -0.7847148776054382, -0.4108307659626007, -0.1305859386920929, -1.161800742149353, 0.07242454588413239, 0.755232572555542, -0.8680887818336487, 0.37256672978401184, 0.2853926420211792, -0.3820109963417053, -1.6585824489593506, -0.35591965913772583, -0.5854420065879822, -1.0893439054489136, -1.1273350715637207, 1.4073814153671265, -0.4316684603691101, -0.36657223105430603, -0.9740328192710876, 0.255430668592453, 0.29121121764183044, -0.44517213106155396, 0.01950880140066147, -1.4540172815322876, 0.25095242261886597, 0.10836750268936157, 0.43505120277404785, -0.37708497047424316, -0.31587666273117065, 0.4007258117198944, 0.06461723148822784, -1.455387830734253, -0.21546036005020142, 2.2254586219787598, 0.14968109130859375, 0.45300352573394775, 0.30931511521339417, 0.19984810054302216, 0.12018799036741257, -0.47740739583969116, 0.4736943542957306, -0.11815416067838669, -0.0496886745095253, 0.9683108925819397, -0.5248359441757202, -0.5969782471656799, -0.7865222692489624, -1.8823338747024536, 0.691551685333252, -0.13444304466247559, -1.4280259609222412, 0.7652424573898315, -0.14826194941997528, -0.8669859766960144, 0.6735542416572571, -1.1578880548477173, 0.47017529606819153, -0.25215068459510803, 0.6005781888961792, -0.0855901688337326, -0.08990972489118576, -0.40842506289482117, 0.21517476439476013, 0.48399367928504944, -0.3458971381187439, -0.6103425621986389, 0.859592616558075, -1.1099917888641357, -1.5459096431732178, -0.6121498942375183, -0.48437201976776123, 2.1995368003845215, 0.18087173998355865, 1.1193273067474365, 1.5641920566558838, 0.9721490740776062, 1.582305669784546, -0.7365483045578003, -0.12097430229187012, 1.3352012634277344, 0.3347812592983246, 0.9123610854148865, -2.056133270263672, -1.2049024105072021, -1.1794699430465698, 0.003238143166527152, -0.7106736898422241, 0.9680243730545044, 1.2364118099212646, 1.9686466455459595, -1.9264063835144043, -1.6567403078079224, -0.43018054962158203, 0.3116871416568756, 0.673994779586792, 0.5330225229263306, -0.45388710498809814, 0.575931191444397, -0.5233099460601807, 1.4928885698318481, -0.8368155360221863, 1.5384832620620728, -0.7628743648529053, 0.02853449434041977, -0.6821351051330566, -0.6404984593391418, 0.18450438976287842, -0.39780691266059875, 0.7526237368583679, 1.2056927680969238, 0.12848667800426483, -1.198923110961914, -0.275105357170105, -0.26381683349609375, 0.5153640508651733, -3.2221486568450928, 0.34119921922683716, -0.586456835269928, -0.1380920708179474, 1.1807199716567993, 1.0296854972839355, -0.865328311920166, -0.2395581752061844, 0.2687009274959564, 0.48775339126586914, 0.3585597276687622, 2.149069309234619, 1.4292885065078735, 2.1237404346466064, -0.9003887176513672, 0.9919434189796448, 2.0505735874176025, -1.4975333213806152, 0.35395005345344543, -0.17157991230487823, -0.22959691286087036, 0.09799293428659439, -1.3603681325912476, -0.3920672833919525, -1.0131367444992065, 0.4452989995479584, 0.8056071996688843, 1.3113524913787842, -1.3436754941940308, -0.08775103837251663, 0.10728035122156143, -0.8167805075645447, 0.7915171980857849, 0.08557527512311935, -0.5816675424575806, 0.9581862688064575, 0.17416897416114807, 2.0473904609680176, -1.0336157083511353, -0.12619638442993164, -0.09692298620939255, 0.9333094358444214, -1.2236943244934082, 1.348400354385376, -2.1373114585876465, -0.08201637864112854, 0.2576672434806824, 1.173744797706604, -0.5241755247116089, 1.0742707252502441, -1.087528109550476, 0.19666756689548492, -0.45015397667884827, 0.170379638671875, -1.8163361549377441, 0.13533924520015717, -0.05031481012701988, -0.8975780606269836, 0.6996506452560425, -1.8525161743164062, 0.8280647397041321, 0.10062398761510849, 1.3316988945007324, 1.3149840831756592, -0.9416857957839966, -0.16352839767932892, -1.1157485246658325, -2.760375738143921, 0.40829336643218994, 0.938590407371521, -0.6711947321891785, 0.9923436045646667, -0.9998350143432617, -0.1877843588590622, -1.5033259391784668, -1.5172432661056519, -1.728302001953125, -0.5537432432174683, -0.2214297503232956, -2.2224059104919434, -0.09029029309749603, -1.008374810218811, -0.8205336332321167, -1.59083890914917, -2.3687903881073, 0.35884639620780945, -0.7410013675689697, -0.4988681674003601, 0.09431417286396027, -0.10110130161046982, 0.0850658118724823, 0.7295966744422913, 1.1123113632202148, 0.11326141655445099, 0.48599478602409363, 1.7624711990356445, -0.6031332015991211, 1.0822240114212036, -1.762315273284912, -0.17406456172466278, -1.749412178993225, -0.9765272736549377, -0.21899887919425964, 0.2018485814332962, -0.479733943939209, 0.6226561069488525, 0.258780837059021, 0.9109134674072266, -2.079528570175171, 1.275232195854187, -0.2630736231803894, -1.1820913553237915, 0.16872546076774597, 1.3480956554412842, 0.06383764743804932, -1.3986611366271973, 2.3909037113189697, 0.12511861324310303, 1.2900104522705078, -0.23114684224128723, 0.13165731728076935, 0.7498293519020081, 1.8343939781188965, -0.26909884810447693, 0.2247144877910614, -0.264060914516449, -1.1087015867233276, 2.207242250442505, 0.9964534044265747, -0.5132798552513123, -1.2006622552871704, -2.047434091567993, 0.2439134269952774, -0.35869160294532776, 0.17465761303901672, 1.350212574005127, 1.0902942419052124, -0.37684351205825806, -1.5523802042007446, -1.169289469718933, 0.15223386883735657, -0.9863013029098511, -0.931520938873291, 1.3861939907073975, 1.222963809967041, -0.20404836535453796, 0.5093822479248047, 1.2196927070617676, -0.056695256382226944, 0.8709540963172913, -0.16553600132465363, 0.8549098968505859, -1.5089119672775269, -0.7535465359687805, 1.675267219543457, 1.2285352945327759, -1.677322268486023, 1.0016266107559204, 0.22274275124073029, 0.46044787764549255, -0.1541258692741394, 2.163055896759033, -0.8234860897064209, 0.7393256425857544, 1.0743175745010376, -1.3491265773773193, 0.10547729581594467, 1.2578198909759521, -1.1373252868652344, -0.431694895029068, 0.24257566034793854, 1.3588923215866089, 0.7249020338058472, -1.351569414138794, -0.8411006927490234, -1.244544506072998, 1.722800612449646, -0.7355706691741943, -0.30500537157058716, 0.5686469078063965, 0.6370342969894409, 0.5660752058029175, 1.5211950540542603, 0.7438058257102966, -0.598447322845459, 0.22442726790905, 0.9965624213218689, 0.3711368143558502, 0.534903347492218, 0.6317334175109863, -0.7736808657646179, 0.24903139472007751, -1.80730402469635, 0.09396066516637802, 0.8998859524726868, -1.726575255393982, -0.6136344075202942, -1.3028439283370972, -0.09589514881372452, 1.6493250131607056, 0.17272470891475677, -0.33414706587791443, 0.3003996014595032, -0.23218245804309845, 0.6300479173660278, 0.30186930298805237, -1.4238202571868896, -1.1751683950424194, -1.3998281955718994, 0.49817007780075073, -1.2790535688400269, 0.7874990701675415, -0.14624474942684174, -2.0414044857025146, 0.17419293522834778, 0.9490195512771606, 0.21987515687942505, -1.300649881362915, 0.059984125196933746, 2.0309834480285645, 2.7389495372772217, -0.7723822593688965, -0.09243295341730118, -0.09384118765592575, 0.9493151307106018, 1.598815679550171, 0.07411005347967148, 0.17980089783668518, -0.7922071814537048, -1.2806850671768188, 0.5109079480171204, -1.6873795986175537, -1.004307746887207, -0.3085866868495941, 0.2688746750354767, -1.9983543157577515, -0.013729131780564785, -0.12739017605781555, 0.02557918056845665, -0.3960508108139038, 0.4955671727657318, 1.1846786737442017, -0.7513489127159119, -1.0841339826583862, -0.5154614448547363, -1.4384243488311768, -0.8197440505027771, 0.4314754605293274, 0.39581170678138733, -0.5837257504463196, -1.5496970415115356, 0.6232649683952332, 0.39301756024360657, 0.4813677966594696, 0.7474867701530457, 1.5571638345718384, 0.9635703563690186, -0.9579484462738037, -3.0276949405670166, 0.15941232442855835, 0.47759515047073364, -0.37032991647720337, -0.11276762187480927, 0.6908585429191589, -0.1313672959804535, -0.08270827680826187, 0.8625262379646301, 0.3502541780471802, -0.3214867115020752, 0.7890028357505798, 0.21798327565193176, 0.4282107651233673, -0.1586945652961731, 0.1537509262561798, -1.7891592979431152, -0.22620023787021637, 1.6270965337753296, 2.376352310180664, 1.7831544876098633, -0.5090544819831848, -1.7266829013824463, 0.6687688827514648, 0.9989293217658997, 1.7673205137252808, 1.2048797607421875, 0.0753130316734314, 1.2763928174972534, 0.1519092470407486, 0.23507201671600342, -1.3450226783752441, 0.44932401180267334, 1.2932219505310059, -0.9252811670303345, 0.21613429486751556, -1.6674987077713013, 0.9294559955596924, 0.07572603970766068, -0.5325884819030762, 0.8644187450408936, 0.3952357769012451, -0.6164603233337402, 0.7508222460746765, -0.4986242949962616, 1.4373412132263184, 0.8586351871490479, -0.810639500617981, -0.9836915731430054, -0.9542380571365356, -0.8138303756713867, -0.6466758847236633, 1.6178678274154663, 0.13344822824001312, -1.7005590200424194, 0.41537249088287354, 0.3652264475822449, 1.4604212045669556, 0.10977937281131744, -0.9049237966537476, 0.42742303013801575, 0.6227648854255676, -1.3599156141281128, -0.3250763416290283, -1.0505759716033936, -0.10518239438533783, -0.45745301246643066, 0.3357955515384674, -1.2573862075805664, -1.5607576370239258, -1.5045154094696045, -0.07729931175708771, 2.3962440490722656, 0.9474709033966064, 0.6738431453704834, -0.4998401999473572, -0.27853724360466003, -0.3076925277709961, 1.4490693807601929, 2.1666953563690186, 1.692165493965149, 0.5262580513954163, -0.730433464050293, -0.9185216426849365, 0.1118289977312088, 0.3676657974720001, 0.41897886991500854, -0.44667455554008484, 0.1881234049797058, -2.5856869220733643, 0.69227534532547, 2.0604803562164307, 0.5354119539260864, 1.5288149118423462, 0.8914932608604431, -0.9194335341453552, -2.5319738388061523, 0.14063043892383575, 0.3205474615097046, -0.6862579584121704, -0.7355333566665649, 0.98637455701828, 0.4454843997955322, -1.5699191093444824, -2.5521953105926514, 1.193650245666504, -0.18130984902381897, 0.8514362573623657, -0.0562039390206337, -0.9861286282539368, -0.009117105975747108, -0.3829421401023865, -1.0440081357955933, -0.8752825856208801, -1.0521022081375122, 1.0094592571258545, 0.1675511598587036, 0.2740843594074249, -0.9240069389343262, 0.12146571278572083, 0.3742501139640808, 1.4033184051513672, 0.30865222215652466, 0.5870806574821472, -0.25906121730804443, 0.7300698161125183, -1.235588788986206, -1.762946367263794, -1.1148972511291504, -0.06591979414224625, 1.0504463911056519, 0.8001533150672913, -1.2263487577438354, 0.22176140546798706, -0.6034464240074158, -1.2401574850082397, -0.6079255938529968, -1.450408697128296, 0.15055310726165771, -0.4376417100429535, -0.972853422164917, -0.48961570858955383, -0.30544736981391907, -0.5934263467788696, -0.6007246971130371, -0.7861972451210022, 0.6404554843902588, -1.2041587829589844, -1.5066457986831665, 0.2229807823896408, 1.6540734767913818, 1.8756457567214966, 1.8113771677017212, -1.0613229274749756, -0.2552986741065979, 0.36449286341667175, 0.3477588891983032, -0.8030665516853333, -0.628147542476654, 1.1398065090179443, 0.5292637348175049, -0.33906856179237366, 0.6185466051101685, 1.3913109302520752, -0.42728492617607117, 1.9766159057617188, -0.7008684277534485, -0.5744202733039856, 0.7494139671325684, 1.3056058883666992, -1.6520521640777588, -0.13007137179374695, 1.3464370965957642, 1.3020215034484863, 0.36371880769729614, -0.4993949234485626, 0.4342939853668213, -0.3717770576477051, 1.5281933546066284, 1.0880465507507324, -0.7494115233421326, 0.8473981022834778, -0.2971879541873932, 0.4317433834075928, 1.8945934772491455, -0.16652466356754303, -1.9411311149597168, -0.043244000524282455, 1.7428704500198364, 1.0680128335952759, -0.2723640203475952, 1.7491304874420166, -0.37319788336753845, -1.2254894971847534, 1.9369171857833862, -0.29397472739219666, -0.34580183029174805, -1.3359044790267944, -0.3842914402484894, 0.4736321270465851, 1.9270721673965454, 1.1814043521881104, 1.295730710029602, 1.238624930381775, -0.7158645987510681, 0.40672698616981506, -0.4178268313407898, -0.6353553533554077, -0.6067423224449158, 0.42404621839523315, 0.8033280372619629, 0.033296212553977966, 0.636626660823822, 1.4336990118026733, 1.0858653783798218, -1.3289557695388794, 1.4010401964187622, 0.32046401500701904, -0.2720121443271637, 0.21921908855438232, -0.4235406219959259, -2.5981431007385254, -0.3916267156600952, 0.4821489155292511, -0.21688270568847656, -0.2190636694431305, -0.41349267959594727, 0.21932511031627655, -1.1164919137954712, -0.3170280158519745, -0.2170625925064087, 0.5142568349838257, -0.5710858106613159, 0.4901738464832306, -0.03890687972307205, 1.9806013107299805, 0.9196071028709412, 0.40063849091529846, -1.100327730178833, 1.889799952507019, -0.5322569608688354, -0.0781850665807724, -0.3425563871860504, -0.3451633155345917, -0.32284843921661377, -1.4730396270751953, -1.299523949623108, -1.1245900392532349, -1.3830589056015015, -0.8070147633552551, -1.0260831117630005, -0.012557798065245152, 1.111978530883789, 1.1367464065551758, 0.4286259710788727, -0.9017488360404968, -0.3342059552669525, 0.19466356933116913, -0.4983808994293213, 1.1925605535507202, 0.9030308723449707, 0.19889940321445465, -0.97882479429245, -1.5937268733978271, -0.2976793050765991, 0.6319114565849304, -1.177106499671936, -0.5404844284057617, -0.982239842414856, 1.390397071838379, 0.343315064907074, -1.0281457901000977, 0.42339855432510376, 0.22484853863716125, 0.7878664135932922, -1.1811237335205078, 1.56596040725708, -0.37842628359794617, 1.9579120874404907, -0.522417426109314, 0.48923245072364807, -0.5384836196899414, -1.105727195739746, 1.264932632446289, 0.7970951199531555, 1.8358708620071411, -0.014377946965396404, -1.0484755039215088, 0.22053024172782898, 0.993385910987854, 0.9036914706230164, -1.091275930404663, 0.2829344868659973, 0.2539718449115753, 0.30767855048179626, 0.7886394262313843, -1.1155905723571777, -1.5619205236434937, -0.4174115061759949, 0.12038792669773102, -2.55249285697937, -0.7910779714584351, 0.9205456972122192, 0.021643728017807007, 0.42400601506233215, 0.5313256978988647, 0.14506150782108307, 0.16323783993721008, 0.40566518902778625, -1.1122866868972778, 0.9115242958068848, -1.5568726062774658, 0.07208608835935593, 0.544096827507019, 0.9714290499687195, -1.3216748237609863, 2.0818052291870117, 0.3576089143753052, 0.2318817526102066, 0.4588451385498047, 0.4662889540195465, 0.29851117730140686, 0.975426971912384, -0.2996511459350586, 0.6739800572395325, 0.1405010223388672, 0.7399235963821411, 0.35886141657829285, -1.0339092016220093, 0.7200334072113037, -0.8108019232749939, -0.6175609827041626, -0.7974209189414978, 0.12130508571863174, 0.23766984045505524, -1.265292763710022, -0.7116509675979614, -0.5122740864753723, 1.0901696681976318, 0.7353851795196533, -1.8899612426757812, 0.15833759307861328, 2.66957950592041, 0.24077987670898438, -0.5202399492263794, 0.61952805519104, 1.664245843887329, 0.17212164402008057, 0.35094577074050903, 1.604540228843689, -1.1328754425048828, 0.9845154285430908, 0.5622959136962891, 0.7405166029930115, -1.3929426670074463, 0.5096549391746521, -1.3610771894454956, -1.3627903461456299, -1.4099761247634888, -1.7614578008651733, -2.085620641708374, 0.9565701484680176, -0.8259119391441345, -0.49358069896698, -1.0187288522720337, -0.5140056014060974, -0.759479820728302, -0.04048988223075867, 1.7557321786880493, -0.010999509133398533, -1.9613133668899536, 0.4711571931838989, 1.5320452451705933, -1.6593958139419556, 0.8554427027702332, -1.7890441417694092, -0.5804992318153381, -1.0542179346084595, -0.8540051579475403, -1.2624305486679077, 0.2416333556175232, 0.04192144796252251, -0.9787344336509705, 0.36472010612487793, 0.8111178874969482, 0.5378583073616028, -0.7585006952285767, -0.20078051090240479, -1.6870783567428589, -0.46733224391937256, -0.29873281717300415, 0.08076083660125732, 0.7500216364860535, -0.34985780715942383, 0.11520235985517502, -0.9008678197860718, 0.8115324974060059, -0.9683926701545715, 1.2321045398712158, 0.1039404645562172, -0.9362806081771851, -0.7129709720611572, -0.9532812237739563, 1.1981643438339233, 0.30855506658554077, 2.0111589431762695, 0.7151842713356018, -1.0170512199401855, 0.7732438445091248, -0.37561333179473877, -0.9417670369148254, 0.4352111220359802, 1.6719540357589722, 1.213715672492981, -0.3472563624382019, 1.0188093185424805, -0.19553059339523315, -0.5235099196434021, 0.9743005037307739, 0.5998550653457642, -0.01263280026614666, 1.2534091472625732, 0.07552552223205566, 1.505603551864624, 0.886390745639801, -0.10682005435228348, -0.2703896164894104, 0.43837133049964905, -1.387253999710083, 0.6670430302619934, -1.3082929849624634, -0.3129236698150635, 0.40675339102745056, -0.01290392130613327, 0.008832089602947235, -3.1922450065612793, -0.9410381317138672, 1.092041015625, 0.6083320379257202, -0.5879037380218506, 1.1426630020141602, 1.1993013620376587, 1.3149652481079102, -1.4724217653274536, 0.05744470655918121, 0.4444187879562378, -0.3153107166290283, -0.032208919525146484, 1.0857923030853271, -0.5105534791946411, 0.3216121196746826, -1.316218376159668, 0.3933139145374298, -0.7641175985336304, 1.7479438781738281, -0.8052705526351929, -0.7962119579315186, 0.4813389480113983, 1.2002431154251099, -0.014619332738220692, 1.6577092409133911, 0.7888511419296265, -3.102721929550171, 1.267250657081604, -1.4972869157791138, 0.1646074801683426, -0.4840807020664215, 0.6772231459617615, 0.16481417417526245, -0.2071712613105774, 0.16148781776428223, 0.7408336997032166, 0.49322909116744995, -0.6080418825149536, 1.616871953010559, -1.4025895595550537, -0.14733785390853882, -2.631711721420288, -0.8638826608657837, 0.2049073576927185, 0.49600258469581604, -0.725260317325592, -1.1832835674285889, 3.032891035079956, 0.5575042366981506, -0.09790212661027908, -0.47201281785964966, 0.9227863550186157, 1.4190019369125366, 0.7600705623626709, 0.932639479637146, -1.2776933908462524, 0.8641364574432373, 0.9169649481773376, -0.7165843844413757, 0.1559639722108841, -1.14740788936615, -2.089451789855957, -0.5120555758476257, -2.3254148960113525, 0.799626350402832, 0.6042197346687317, 0.7238138318061829, -0.5941445827484131, -1.0868386030197144, 0.13350464403629303, 1.096871256828308, 0.8619709014892578, 0.7476937174797058, -0.5853087902069092, -0.30428215861320496, -0.5754411220550537, -0.13866257667541504, 0.17524108290672302, 1.131745457649231, -0.11540885269641876, 0.46703121066093445, 0.9403085708618164, 0.3006938397884369, -1.2146731615066528, -0.8001781105995178, 0.7292353510856628, -0.2556823790073395, 0.3350135087966919, 1.251808762550354, 1.1402866840362549, 0.28415536880493164, 0.7305625081062317, 3.0441129207611084, -0.8511072397232056, -1.0360620021820068, 1.1072672605514526, -1.4723165035247803, -0.8190494179725647, 0.11773623526096344, -0.5276108384132385, -1.794395089149475, 0.8516110777854919, -0.20323418080806732, -0.48282313346862793, 0.3051377832889557, -1.3653801679611206, 0.7111542820930481, -1.1735488176345825, -1.7873103618621826, -0.5606262683868408, 0.09255574643611908, 1.056016206741333, -0.6940005421638489, 0.09362099319696426, 0.6436223983764648, 0.03165523335337639, 0.7915028929710388, 0.3665579557418823, 1.2974640130996704, 0.41196098923683167, 1.5216137170791626, 0.19529008865356445, -0.39365848898887634, 0.3447437584400177, -1.1721059083938599, -0.19368667900562286, -0.38412922620773315, -1.5394887924194336, -0.3137117624282837, -1.838749885559082, -0.4830344617366791, 0.5627946853637695, -0.01565786451101303, 0.861943244934082, -0.4093942940235138, -0.7053495645523071, -1.7780698537826538, 0.27127960324287415, 0.27787452936172485, -0.4069898724555969, 0.10608725249767303, -0.13591325283050537, 1.5214388370513916, 0.16250544786453247, 1.1228338479995728, 0.6754258871078491, 0.5525147914886475, -0.8085110187530518, -2.8590238094329834, -2.066141128540039, 0.38268908858299255, -0.0755932629108429, -0.22895710170269012, -0.17254126071929932, 0.9313477277755737, -0.14795297384262085, -0.9287609457969666, 0.6315773725509644, -1.283792495727539, 1.0863806009292603, 0.3885114789009094, -0.4375304877758026, -0.7649354338645935, -0.13314148783683777, -1.270695686340332, 0.7936983704566956, -1.72835373878479, -0.8388656973838806, 0.12934623658657074, -1.0938705205917358, 0.20283350348472595, 1.5247348546981812, -1.5350995063781738, -1.5648130178451538, -0.7569870352745056, -0.3246607184410095, 0.6123788952827454, -1.1620399951934814, -0.6081159114837646, -0.5812810659408569, -0.2764727771282196, 1.0249332189559937, -0.8881368041038513, 0.5472431778907776, -0.07190840691328049, -0.09893617779016495, -0.0222813431173563, -0.762816846370697, 1.3742741346359253, -0.4444630444049835, -1.0822514295578003, 0.5534709095954895, -0.02343306504189968, -0.3677966594696045, -1.9099053144454956, -0.44793063402175903, -0.47665318846702576, -1.7030354738235474, -0.5893360376358032, -0.8160653710365295, 0.08324577659368515, -2.1742751598358154, -1.4818811416625977, -1.0563633441925049, 0.7830380201339722, 0.12516732513904572, 0.37345436215400696, 0.9109508395195007, 1.6805803775787354, 0.9069501757621765, 0.9437510371208191, -1.6634854078292847, 0.3911854326725006, 0.7513353824615479, -1.7528349161148071, -0.8176016807556152, 0.9857856035232544, 1.466046929359436, 1.3654810190200806, 0.773966908454895, 0.8872935175895691, -0.7859294414520264, 0.3583597242832184, -1.076961636543274, -1.4709022045135498, 1.5996170043945312, -1.8188389539718628, -1.0001933574676514, 0.7835549116134644, -0.8528231382369995, -0.48254528641700745, 1.1820939779281616, 1.479875922203064, 0.5477403402328491, -0.4453127980232239, 0.1513652503490448, 0.24677668511867523, 0.5245550274848938, -0.7488676905632019, -1.784736156463623, 0.07649580389261246, -1.237395167350769, -0.37131017446517944, 1.143320918083191, -1.2328928709030151, 0.7205938696861267, -1.1692994832992554, -0.6126574873924255, -0.6334496736526489, 0.9931711554527283, -0.737940788269043, -0.68277907371521, 0.045624084770679474, -0.4510292410850525, 1.1630523204803467, 0.5986695289611816, 0.09963718056678772, -0.0813574343919754, -0.8880906701087952, 2.505911111831665, 0.22168013453483582, 0.018306607380509377, -2.1772618293762207, -0.5299527049064636, 1.5407867431640625, -1.5991157293319702, 1.0498888492584229, 0.16288159787654877, -0.1451840102672577, 0.4282139241695404, 0.3949582576751709, 1.8396302461624146, -1.1784056425094604, -0.9552372097969055, -0.6334726810455322, -0.9299506545066833, 1.1447449922561646, 0.7929144501686096, 1.106429934501648, 1.015108346939087, 0.12210705876350403, -1.0400869846343994, -0.6839567422866821, -0.276268869638443, 0.3609349727630615, -0.1947300136089325, 0.22900502383708954, -0.6734551191329956, 1.676867127418518, -0.5844523310661316, 1.8625478744506836, -0.7401322722434998, 0.09641730040311813, 1.4588863849639893, -1.7089564800262451, 1.1210343837738037, 1.2840932607650757, -1.1670995950698853, 1.3540247678756714, 0.09554041922092438, 0.8248714804649353, 0.5357593894004822, -0.1123165413737297, -0.8071437478065491, 0.5593304634094238, -1.6905810832977295, -0.09218447655439377, 0.8369698524475098, 0.6382046937942505, 0.05138322710990906, 1.581575870513916, 0.8545025587081909, -0.2794179618358612, -0.7524628043174744, -2.2889459133148193, -0.468820720911026, 1.1980952024459839, -0.8752856850624084, 0.30366477370262146, -0.07445932179689407, -0.6617286205291748, 1.009223222732544, 0.36441341042518616, 2.151143789291382, -0.7513260245323181, -1.0802339315414429, -0.472007691860199, 1.6849342584609985, -1.582395076751709, 1.0643161535263062, 0.24456746876239777, -0.19060343503952026, -0.3462497293949127, 0.9832834601402283, -1.3077051639556885, -0.28118541836738586, 0.24369266629219055, 0.5253918766975403, -0.19714772701263428, 0.8706505298614502, 1.8254348039627075, 0.5084410905838013, 0.3100872039794922, -0.006603511516004801, 0.6485028266906738, 0.5702083110809326, -0.3895309865474701, -0.9494929313659668, 1.5026302337646484, -0.8263075947761536, -0.7863596081733704, 0.3302091658115387, -1.86764657497406, 1.400169014930725, 0.19450309872627258, -0.35311099886894226, -0.4058728814125061, 0.955742597579956, 0.29499971866607666, 0.5781583189964294, 1.3913798332214355, 0.2531471848487854, -0.2792275846004486, 0.24470163881778717, -2.015897274017334, -1.2461738586425781, 0.7061097621917725, 0.07957857102155685, -0.4200364351272583, 0.48279404640197754, -1.5188484191894531, 1.5057880878448486, 1.052276372909546, -0.7783272862434387, 1.681656002998352, 0.033033594489097595, -1.1496161222457886, -0.9056341648101807, 0.0034499613102525473, 0.764300525188446, 0.1741621345281601, 0.6329178214073181, -0.12154480814933777, -0.9637216329574585, 1.3171377182006836, 0.9344584941864014, -1.2503234148025513, 0.37439826130867004, -0.8197271823883057, 0.24020637571811676, -0.5593081712722778, 0.835300624370575, -0.24414227902889252, -0.8628912568092346, -0.634290874004364, 0.030080169439315796, -0.676952064037323, -0.6840973496437073, 1.1798419952392578, 0.7660253643989563, -1.5256407260894775, -0.5069798231124878, -0.8644242882728577, -1.9847620725631714, 0.7125492095947266, -1.788121223449707, -1.0156548023223877, -0.9347097873687744, 1.5505748987197876, 1.7957663536071777, -0.19244049489498138, -0.844899594783783, 0.1574472188949585, 1.56455397605896, -1.1971315145492554, 1.0833110809326172, 0.3852173984050751, -1.5652481317520142, 1.3270667791366577, 1.129472255706787, 0.3563319146633148, 0.11162693053483963, -0.10999329388141632, -0.6486575603485107, -2.2149055004119873, -0.32910943031311035, -1.382689356803894, -0.3129901587963104, -0.7663793563842773, 0.24150343239307404, -0.8819335103034973, 0.9399164319038391, -0.24647311866283417, -0.10214857757091522, 0.13034114241600037, 0.1961383819580078, -0.28189119696617126, 0.7824955582618713, 1.4325268268585205, -1.231052041053772, 0.7965205311775208, 0.2191760241985321, 0.49494820833206177, -0.4452676475048065, -0.8461926579475403, -0.1349470466375351, 0.6580338478088379, 0.30689045786857605, -0.3409947454929352, 0.4654249846935272, 1.5396941900253296, 1.2371374368667603, -1.9716317653656006, 1.2334764003753662, -0.9538540840148926, -0.03276190161705017, 1.000598430633545, 1.219998836517334, 0.5633249878883362, 0.5696370005607605, 0.6241642236709595, -0.7608382105827332, -0.6875640153884888, -0.9036887288093567, -0.44298553466796875, 0.8123335242271423, 0.5553683042526245, 0.4799639582633972, -0.2301385998725891, -1.495195746421814, 0.8753924369812012, -0.531805694103241, 0.5037875771522522, 1.0866281986236572, 0.8227593302726746, 1.0223476886749268, -0.7098991274833679, 0.35687685012817383, 2.4023215770721436, 2.795233726501465, -0.4579087197780609, 2.1761860847473145, -0.5556601285934448, -0.22920264303684235, 1.7764440774917603, -2.2533974647521973, 0.4759484827518463, 0.36773523688316345, -0.09503690898418427, 0.2020644098520279, 1.83900785446167, -1.0206024646759033, -0.3548262417316437, 0.7101584076881409, 0.5957857966423035, -0.7654139399528503, 0.00594399543479085, -0.631283164024353, -0.3383803367614746, -1.0499298572540283, 0.946625828742981, -0.342073917388916, 1.2987149953842163, 0.5651044249534607, 0.028532473370432854, 0.013591364957392216, 0.3320761024951935, -1.441717505455017, 1.6647030115127563, -0.640985369682312, -0.8400087952613831, 0.06926160305738449, -0.8989502191543579, -0.20021989941596985, 1.4693071842193604, 2.7210397720336914, 0.43031445145606995, -0.5358967781066895, -0.4845477044582367, -0.11407513916492462, -0.7874804139137268, 1.8661870956420898, -0.46672019362449646, -0.2811447083950043, -1.0238397121429443, -0.2860751450061798, 0.7539564371109009, -0.17053742706775665, 2.00515079498291, 2.349689245223999, -0.7861565351486206, -0.9648792743682861, -0.9968997836112976, -0.8233811855316162, 2.9342429637908936, -0.35162463784217834, 0.3052816689014435, 0.36369121074676514, -0.3915083408355713, -3.3524227142333984, 0.9509153962135315, -1.0016118288040161, 2.4424476623535156, -1.055917739868164, -1.5786041021347046, -1.9658294916152954, -0.3545500338077545, -0.2431616634130478, 2.208482265472412, 0.41645902395248413, -1.204959511756897, 0.4868258833885193, -0.4853270351886749, 0.2226979285478592, -1.5553786754608154, -1.9719759225845337, 0.9443421959877014, -1.3082295656204224, 0.7810997366905212, 1.5608991384506226, -1.5432034730911255, 1.5560197830200195, -0.0928516760468483, 1.1841286420822144, 0.9503482580184937, -0.19149596989154816, 1.2760264873504639], "dtype": "float32"}], "e": -3.3524227142333984}, {"a": [{"__nd__": [-70.55450439453125, 40.11592102050781, 6.8898186683654785, 22.934391021728516, -2.3090219497680664, 93.42512512207031, -22.294727325439453, 31.696821212768555, 71.58959197998047, 30.88601303100586, 6.966533660888672, 49.83664321899414, -12.954944610595703, 54.78494644165039, -60.69122314453125, 4.967753887176514, 27.06808090209961, 14.520882606506348, 43.564605712890625, -25.42466926574707, -17.62287139892578, -34.244651794433594, -4.591182708740234, 1.3879446983337402, -10.12740421295166, -8.147050857543945, -68.47799682617188, 19.066890716552734, -11.042259216308594, -44.53584671020508, 45.419986724853516, -19.528079986572266, 43.133934020996094, -53.30967712402344, 138.1361846923828, 28.9028263092041, -3.6499290466308594, 78.14000701904297, -10.92424201965332, 24.503154754638672, -42.072349548339844, 72.004150390625, 3.6968576908111572, 31.671730041503906, 29.351795196533203, 48.41256332397461, 62.55223846435547, -117.84709930419922, -49.16387939453125, 102.35118865966797, 81.33088684082031, -14.104065895080566, 32.23878860473633, 80.97794342041016, 30.171783447265625, 81.08519744873047, 32.8863639831543, 103.58531951904297, 77.3791732788086, -60.77496337890625, -2.5196855068206787, 12.262723922729492, 83.43826293945312, -52.10618591308594, -77.8704605102539, -29.858692169189453, 60.04008102416992, 93.4245834350586, 9.198150634765625, 14.595550537109375, 19.91013526916504, -7.427590370178223, 68.07746887207031, 79.89667510986328, -59.31378173828125, 29.846797943115234, -13.284743309020996, -45.713619232177734, 32.48214340209961, 18.932039260864258, -0.6789448261260986, -34.711700439453125, -25.630949020385742, 4.1824822425842285, 57.489349365234375, 68.62963104248047, -22.6619930267334, 36.747188568115234, 101.59439086914062, -74.7907943725586, -2.6592724323272705, -1.702209234237671, -9.253373146057129, 61.56051254272461, 9.190143585205078, 37.818241119384766, 20.093523025512695, 78.6650619506836, 74.13653564453125, 135.20654296875, 26.52466583251953, 53.91562271118164, -18.523696899414062, 7.296558856964111, 77.93547821044922, 73.68958282470703, 73.50684356689453, 7.315052509307861, 27.76640510559082, 14.31842041015625, 10.212995529174805, 41.7198371887207, -20.636873245239258, 45.36688995361328, -3.268303155899048, -45.16122055053711, 51.04172134399414, -16.282901763916016, 48.725914001464844, -15.709970474243164, -28.281883239746094, -14.480931282043457, -31.093040466308594, 25.06755828857422, -0.7730435729026794, 1.732622504234314, -33.806251525878906, 47.015296936035156, 44.176246643066406, 50.48401641845703, -7.643805503845215, -16.52986717224121, 29.0372257232666, 19.56369972229004, 55.538211822509766, 36.26331329345703, 9.678752899169922, -26.360754013061523, -3.540008068084717, -17.133983612060547, 131.3609619140625, 52.49485778808594, 108.13713836669922, 17.24752426147461, 41.403663635253906, -21.51112174987793, 35.63910675048828, -14.692809104919434, 76.57064056396484, -29.297616958618164, 49.187828063964844, 60.69184494018555, 50.54335403442383, 104.20121002197266, -26.639524459838867, 7.137595176696777, -67.14557647705078, -40.88011932373047, -53.54062271118164, -25.503780364990234, 47.71161651611328, 41.291786193847656, -66.09890747070312, -40.227203369140625, 93.26242065429688, 16.751346588134766, 41.0487174987793, -53.59109878540039, 21.149002075195312, 144.7144012451172, 2.2874746322631836, 21.757841110229492, 7.605197906494141, 38.80403137207031, -5.487109184265137, -50.809242248535156, 78.53608703613281, 56.88471984863281, -42.8470573425293, 69.10541534423828, 77.11428833007812, 63.81922912597656, 71.11260986328125, 128.3578338623047, -22.268550872802734, -31.49828338623047, -1.6960526704788208, 54.91865921020508, 62.9943962097168, 10.271408081054688, 75.78480529785156, -37.602142333984375, 34.36071014404297, -1.0621578693389893, -40.38157653808594, -54.228271484375, 55.01571273803711, 12.759052276611328, 40.74720001220703, 21.55147361755371, 126.02108764648438, 78.86504364013672, -7.082600116729736, 112.7774887084961, 19.974666595458984, 10.546676635742188, 67.28987884521484, 46.64890670776367, 32.478328704833984, -37.074134826660156, 66.51007843017578, 11.605207443237305, -54.16104507446289, -18.51438331604004, -107.79082489013672, 8.366644859313965, 26.9454402923584, 11.941879272460938, 3.613506317138672, -47.44906997680664, 84.8858871459961, 110.19061279296875, 22.581506729125977, 40.998165130615234, 45.26797103881836, 148.6358642578125, 79.34089660644531, 15.99765682220459, 83.96710205078125, 39.54146194458008, -23.50990867614746, 39.18513107299805, -46.04468536376953, 103.23823547363281, -27.79617691040039, 143.8577880859375, 14.26181411743164, 5.884195327758789, 31.786848068237305, -63.81570816040039, 96.2791976928711, 17.444143295288086, 17.952594757080078, -73.61543273925781, -27.196449279785156, 32.81837463378906, -62.329200744628906, 44.83375549316406, -39.89110565185547, -24.42422866821289, -5.936431884765625, 14.646512031555176, -32.432064056396484, 4.681143283843994, 72.63772583007812, -23.17310905456543, 80.84737396240234, -77.97010803222656, 2.6264045238494873, 64.56729125976562, -23.021244049072266, 14.155957221984863, 17.105405807495117, 70.99835968017578, 92.45265197753906, 35.10232162475586, -29.91656494140625, 118.30768585205078, 34.92805480957031, 26.788976669311523, 5.077274799346924, 40.37589645385742, 15.436471939086914, 32.424808502197266, 41.706275939941406, 106.08116912841797, 8.370613098144531, -75.10420227050781, -31.486658096313477, 26.127239227294922, 15.09749984741211, 31.811304092407227, 108.0157241821289, 107.85904693603516, 22.717859268188477, -0.2276231199502945, 80.1751937866211, 12.164517402648926, 27.166099548339844, 151.19374084472656, -28.326513290405273, 34.724456787109375, 56.33468246459961, 21.595935821533203, 36.459190368652344, -19.055768966674805, 40.470252990722656, 17.384279251098633, -7.769278049468994, 10.561274528503418, 36.73469543457031, 9.281628608703613, -4.552913665771484, 74.84210968017578, 60.95094680786133, 18.01232147216797, 39.160560607910156, 77.51705169677734, -16.414703369140625, 13.027713775634766, 38.305721282958984, -42.994285583496094, 34.76436996459961, -32.425411224365234, 42.42842483520508, -42.06563186645508, 89.64178466796875, -18.398319244384766, 32.92641830444336, 26.45175552368164, 103.20162200927734, 46.34907531738281, -37.83004379272461, -62.42919158935547, -12.437823295593262, -59.86395263671875, 39.436363220214844, 4.237971305847168, 91.12532806396484, -17.7640380859375, 19.27301788330078, 74.00532531738281, 37.8951530456543, 0.3698965609073639, 93.35885620117188, -23.038625717163086, 30.377323150634766, 36.90277099609375, 66.36903381347656, 112.91790008544922, 34.94792175292969, 22.27491569519043, 18.450342178344727, 75.08644104003906, 29.7106990814209, -0.5199995636940002, 64.32913970947266, 55.12348937988281, 21.985370635986328, 63.93828582763672, 4.535192012786865, -16.767133712768555, 27.119956970214844, -33.44709014892578, -12.328731536865234, 142.41749572753906, 70.63451385498047, 123.0120620727539, -35.13850784301758, 29.25037384033203, 5.3267998695373535, 57.748435974121094, 42.7740364074707, 26.909914016723633, 51.42241287231445, 28.336353302001953, -9.98832893371582, 8.620636940002441, 27.386554718017578, 68.53662109375, 113.03176879882812, 67.5270004272461, 46.92234420776367, 98.14323425292969, -18.231470108032227, 16.592370986938477, 22.24222183227539, -28.6588077545166, 60.09014129638672, 42.586700439453125, 52.16542053222656, -21.503263473510742, 16.316078186035156, 24.196147918701172, -26.973962783813477, 54.783241271972656, 62.467247009277344, 51.07990264892578, 17.39490509033203, -14.36428165435791, 42.8592529296875, 46.701541900634766, -5.400155067443848, 21.022510528564453, 44.48214340209961, 18.614139556884766, -15.272560119628906, -98.82627868652344, -49.59428787231445, 6.5313310623168945, 31.171566009521484, -1.996407151222229, 65.52739715576172, -47.59528732299805, 25.71394920349121, -3.215393543243408, 26.66392707824707, -15.971231460571289, 1.0770691633224487, 79.31398010253906, 17.54993438720703, 81.57849884033203, -39.052486419677734, 14.298916816711426, -55.93960952758789, -47.129051208496094, -40.12526321411133, -78.72295379638672, -15.975866317749023, 22.687740325927734, 37.649559020996094, 69.14584350585938, -9.799355506896973, 54.613739013671875, 93.11017608642578, 69.01228332519531, 14.064997673034668, 11.079635620117188, 31.940771102905273, -14.221116065979004, 18.407135009765625, 35.94741439819336, 27.566858291625977, 63.75601577758789, 85.30636596679688, -30.443941116333008, 11.795377731323242, 75.12533569335938, -16.590776443481445, 7.740262985229492, -58.644412994384766, -95.01013946533203, 0.28627046942710876, 40.5031623840332, 85.54509735107422, -34.95948791503906, 75.82772064208984, 91.40756225585938, 6.280782699584961, 34.16572189331055, -48.509422302246094, 34.64886474609375, -2.250864267349243, -20.55796241760254, -36.48064422607422, 71.39643096923828, -37.85026550292969, 32.84904479980469, 13.034939765930176, 28.009416580200195, 128.5789031982422, 26.576810836791992, -57.90351867675781, 39.00017166137695, -11.600017547607422, 84.8100357055664, 81.03800964355469, 9.824353218078613, 121.80023193359375, -100.55416870117188, -24.711538314819336, 30.00602912902832, 23.287290573120117, 32.57509994506836, 1.5444204807281494, 13.124682426452637, -13.717126846313477, 35.608028411865234, 0.9065775275230408, 20.75724220275879, -14.81458854675293, 1.2496839761734009, -34.366424560546875, 85.30393981933594, -29.64206886291504, 78.7604751586914, 94.55884552001953, 9.530932426452637, 47.43654251098633, 105.7313003540039, 13.147621154785156, -2.5032386779785156, 77.0848159790039, 23.406614303588867, -20.514347076416016, 14.529047966003418, 11.291403770446777, 211.0063934326172, 10.755043029785156, 65.10315704345703, 8.59260368347168, 49.6074333190918, -26.06801414489746, 78.94702911376953, 20.12681770324707, 78.71273803710938, 168.45277404785156, -39.61922073364258, 54.480403900146484, 16.897357940673828, -29.89305877685547, 10.442553520202637, 31.107877731323242, 19.968698501586914, 29.233627319335938, 59.706302642822266, 38.77876663208008, 28.93896484375, 12.88433837890625, -12.168524742126465, 40.90461730957031, 40.78099060058594, 32.87834930419922, -11.287396430969238, 42.18427658081055, 52.69140625, 73.13007354736328, 39.47079849243164, 45.942039489746094, -63.60136032104492, 37.834571838378906, 78.3553466796875, 1.505903959274292, 22.61329460144043, 56.513694763183594, -57.51915740966797, 55.31853103637695, 14.829266548156738, -2.8337371349334717, -69.7580337524414, -57.16754150390625, -40.848751068115234, 14.987147331237793, -47.70587158203125, 40.37413787841797, 16.351327896118164, -4.635797023773193, -60.40079116821289, -49.80031967163086, 96.94950866699219, 121.51209259033203, 67.98251342773438, 42.50685501098633, -50.915367126464844, 69.1923828125, -53.494503021240234, -17.2344913482666, 85.79864501953125, 24.043901443481445, 52.11043167114258, -106.22401428222656, 22.734601974487305, 41.07709884643555, 43.619205474853516, 74.47451782226562, 48.7105712890625, 44.56563949584961, 61.77782440185547, 33.362693786621094, 43.99891662597656, 31.778152465820312, 41.70338821411133, -4.930344104766846, 66.06401062011719, -62.245243072509766, 58.320213317871094, 118.08475494384766, 71.38995361328125, 15.038496017456055, -24.77372932434082, 15.214734077453613, -8.334230422973633, 32.351356506347656, 35.459312438964844, -51.64052200317383, 39.543331146240234, 54.18888854980469, 6.594954967498779, -34.537986755371094, -89.15476989746094, 34.79091262817383, 75.59004211425781, 43.72731018066406, -2.8834221363067627, 47.85978317260742, 11.760169982910156, 14.583104133605957, -8.455183029174805, 102.28495788574219, -72.11688995361328, -4.893975734710693, 78.02933502197266, 66.06246185302734, 124.94752502441406, 45.08144760131836, -38.920040130615234, -22.476438522338867, 58.69755554199219, 61.31021499633789, 47.76877975463867, 30.117223739624023, -17.204118728637695, -45.36191177368164, -21.379642486572266, 76.07951354980469, 36.01521682739258, -29.334745407104492, 77.01422119140625, -84.5712890625, 45.792789459228516, -11.891846656799316, 22.43334197998047, 77.72406005859375, 52.05297088623047, 16.76030921936035, 64.17626953125, 56.467281341552734, 23.33734703063965, -4.820175647735596, 81.37500762939453, -12.48402214050293, 142.56982421875, 45.65073013305664, 57.978939056396484, 62.73151397705078, 6.560913562774658, 99.35562133789062, 8.12368392944336, 12.432225227355957, -72.96732330322266, 158.99984741210938, -3.7352895736694336, 86.77226257324219, 36.65354537963867, 36.60057830810547, 68.35993194580078, -60.27992248535156, 31.509065628051758, -40.46013259887695, 49.96709442138672, 26.334630966186523, -10.090181350708008, 76.48687744140625, 9.963579177856445, 47.65391159057617, 90.58187866210938, 77.8491439819336, 38.9876708984375, -14.027756690979004, 110.11312103271484, -43.13994216918945, 27.820709228515625, 103.36202239990234, 27.85552978515625, 8.4764404296875, 39.40346908569336, 40.007755279541016, -9.103946685791016, 47.76587677001953, -2.3389029502868652, 74.32035827636719, 40.636837005615234, -56.80278015136719, -15.492130279541016, 39.38133239746094, 67.59831237792969, -64.6934814453125, -3.4768102169036865, 20.1560115814209, 84.25323486328125, -58.384281158447266, 22.309608459472656, 50.272743225097656, 12.147858619689941, 59.79435729980469, -49.22967529296875, -63.343833923339844, -34.38922119140625, 67.0623550415039, 20.457521438598633, 95.5052261352539, 18.085140228271484, -31.406349182128906, 45.26224899291992, 46.60003662109375, -5.888053894042969, -44.69190979003906, -48.001853942871094, -27.563289642333984, -2.2137651443481445, 39.44509506225586, 26.59883689880371, 59.25467300415039, -47.382198333740234, -27.629817962646484, 48.29148483276367, 45.741455078125, 30.583852767944336, 39.89716720581055, -30.776365280151367, -53.36038589477539, -5.942418575286865, 109.91856384277344, 45.673336029052734, -8.321895599365234, 113.55200958251953, -47.44438552856445, 28.3072452545166, 55.904945373535156, 44.57575607299805, 52.66628646850586, 8.181632041931152, 33.42091751098633, -33.76541519165039, 29.79384422302246, 55.627159118652344, 52.43050003051758, 71.66156768798828, -58.35372543334961, -31.100364685058594, -42.050987243652344, 40.32588195800781, -72.54510498046875, -12.020015716552734, 18.69657325744629, 94.5339126586914, 48.2684326171875, -23.537189483642578, -45.810157775878906, -31.47376823425293, -27.73939323425293, 8.563551902770996, 93.75687408447266, 19.166126251220703, 13.367291450500488, 11.661209106445312, 19.206144332885742, 32.30751419067383, 40.4896240234375, 100.34486389160156, 94.4407730102539, -0.9639483690261841, 25.209552764892578, 61.38562774658203, 103.42125701904297, 11.475316047668457, 44.780147552490234, -20.300945281982422, -20.35285758972168, 9.852776527404785, 8.467706680297852, 7.266252517700195, 70.35606384277344, 63.421077728271484, 3.6452083587646484, 121.0656509399414, 4.860171794891357, 25.601688385009766, 68.70974731445312, 45.69602966308594, 22.2572021484375, 41.22052001953125, -45.33316421508789, -3.1174964904785156, 158.803955078125, 55.39482116699219, 90.48197937011719, 103.54200744628906, -12.18931770324707, 45.64900207519531, -2.558339834213257, -80.41107177734375, 50.103355407714844, 8.249518394470215, 41.0120964050293, -2.04785418510437, 69.85928344726562, 61.89742660522461, 64.91922760009766, -28.422504425048828, -30.512409210205078, 61.3287467956543, 50.814964294433594, 6.411325931549072, 87.7805404663086, -71.4586410522461, -63.62901306152344, 87.29871368408203, 8.615445137023926, 3.1609582901000977, 36.8543701171875, 40.95309066772461, -57.17387008666992, 55.723812103271484, 8.625874519348145, 38.44718551635742, -3.194272041320801, 71.02662658691406, 77.55187225341797, 65.11619567871094, 48.82439422607422, 78.61286926269531, 50.130672454833984, -17.31680679321289, 94.18061065673828, 34.18349838256836, -16.51289176940918, 12.507725715637207, 143.71246337890625, 115.26250457763672, 55.494747161865234, 1.6341253519058228, 72.08185577392578, -92.22570037841797, 4.783578395843506, 146.99667358398438, 38.60261535644531, -20.87644386291504, 72.65833282470703, 175.86827087402344, -18.2918701171875, 35.626033782958984, 61.54262924194336, -59.5567626953125, 32.694679260253906, -9.716202735900879, -28.352821350097656, 11.62535572052002, -15.531966209411621, -30.739896774291992, -34.176910400390625, 24.94638442993164, 13.218016624450684, -65.47561645507812, 56.41761016845703, -16.443283081054688, 65.68318176269531, 42.370182037353516, -54.44577407836914, -32.082794189453125, -4.814482688903809, 19.285581588745117, -8.393440246582031, 42.87232971191406, -14.492918014526367, 46.09151840209961, 10.690607070922852, 58.068084716796875, 82.91519927978516, 51.94264602661133, 47.204917907714844, 23.878232955932617, 101.26704406738281, 108.52796173095703, 93.72339630126953, 9.22904109954834, -10.985156059265137, 81.25923156738281, 30.939889907836914, 27.378759384155273, 48.82669448852539, 16.674394607543945, -84.04356384277344, 10.941972732543945, 68.5450439453125, -4.858438491821289, 58.01156997680664, -43.82564163208008, 82.14080810546875, 12.862334251403809, 65.23466491699219, 56.192779541015625, 33.72166442871094, -3.3002676963806152, 100.72563934326172, 17.022571563720703, 16.661258697509766, -10.489583015441895, 87.17149353027344, -15.11188793182373, 70.13177490234375, 67.95777130126953, 59.704227447509766, 102.87870025634766, -29.77519989013672, -64.18476104736328, 55.626548767089844, -34.33076477050781, -41.13361740112305, -21.308223724365234, 17.751853942871094, 19.215255737304688, -2.0350866317749023, 74.572265625, 19.09200096130371, -21.962966918945312, 63.27101516723633, -70.51005554199219, 3.965951442718506, 6.005444049835205, 40.15007781982422, 3.4584858417510986, -24.275421142578125, 101.55886840820312, 8.608430862426758, 6.26362943649292, 8.207460403442383, 91.08828735351562, -16.668567657470703, -21.631811141967773, 31.15418243408203, 30.900508880615234, 19.194520950317383, 5.81599760055542, 48.65729904174805, -3.492729663848877, -8.375825881958008, 177.08741760253906, 18.249351501464844, -26.388818740844727, 121.31349182128906, 2.6980667114257812, 2.69049072265625, 58.09797668457031, 146.97445678710938, 66.29578399658203, -12.800018310546875, -48.15562438964844, 67.87476348876953, 51.86739730834961, 67.54061126708984, 56.51279067993164, 94.84703826904297, 25.018131256103516, -28.70936393737793, -79.07465362548828, -24.975967407226562, -39.77848815917969, -37.278141021728516, 5.2723517417907715, 35.93726348876953, 4.593170166015625, 47.106754302978516, 128.4494171142578, 42.42572021484375, -111.60946655273438, 10.544862747192383, 28.509281158447266, -42.395965576171875, 49.71031951904297, -11.535444259643555, -61.83131408691406, 52.37763977050781, 59.505943298339844, -34.875362396240234, 46.32637023925781, -19.122894287109375, -67.11852264404297, -10.375445365905762, 132.66262817382812, 24.76104736328125, 61.48427963256836, 98.77252197265625, -21.757112503051758, 22.865793228149414, 24.11553382873535, 54.266666412353516, -18.8288516998291, 20.157020568847656, 49.99757766723633, 101.48497009277344, 14.813176155090332, 139.4523162841797, -66.54651641845703, 100.65321350097656, 34.4387092590332, 47.43418502807617, 37.04791259765625, 14.203513145446777, 6.400494575500488, 52.5531120300293, 51.95768356323242, -63.253475189208984, -61.38692092895508, 0.6777763366699219, 83.77254486083984, 23.870811462402344, -44.22639465332031, -43.68488693237305, -84.10292053222656, 85.99334716796875, 130.60533142089844, 20.817195892333984, 130.5576171875, 91.63375854492188, 27.778657913208008, 40.93141555786133, -41.17782974243164, 5.4481892585754395, 3.2686402797698975, 71.18035125732422, -17.221920013427734, 42.53459548950195, -0.7901707291603088, 68.72142028808594, 79.16531372070312, -61.58904266357422, -77.1189193725586, 123.4699478149414, -51.12981033325195, 28.091060638427734, -13.016593933105469, 31.32868194580078, -71.00858306884766, 92.51380157470703, 42.164756774902344, 142.68133544921875, -5.657619476318359, 33.89459228515625, -9.761354446411133, 16.42967414855957, -80.56922912597656, -18.892580032348633, 54.30633544921875, 9.439203262329102, -10.102323532104492, 16.319778442382812, 27.86067771911621, 110.56095123291016, 46.92341995239258, -26.91594696044922, -8.272144317626953, -15.987730026245117, -13.962770462036133, -36.99229049682617, -15.249067306518555, -58.51420593261719, 101.23503875732422, 76.08998107910156, -44.470115661621094, 59.73822784423828, -40.76814270019531, 20.03849220275879, 101.47791290283203, -43.36448669433594, -39.20745849609375, 56.19093704223633, 20.919580459594727, -25.170366287231445, 24.346996307373047, 19.346824645996094, -14.160965919494629, 14.456615447998047, 96.59333038330078, -18.776779174804688, 32.09830093383789, -5.430741786956787, -85.8362045288086, 50.952911376953125, 0.911577045917511, 14.274145126342773, -27.51944923400879, -43.875762939453125, -60.316001892089844, 36.98542785644531, -11.701881408691406, 41.708656311035156, -1.8308497667312622, 118.13042449951172, 77.10049438476562, 48.120849609375, 31.584571838378906, 52.227569580078125, 20.811769485473633, 60.647865295410156, 57.89638900756836, -8.522016525268555, -16.83762550354004, -27.082834243774414, -1.2325893640518188, -5.49293851852417, -13.686628341674805, -7.087185382843018, 37.12893295288086, 38.6785774230957, -20.28761100769043, -4.728851318359375, -22.189186096191406, -10.148005485534668, 45.52864456176758, 9.918081283569336, -59.96231460571289, 30.587522506713867, 60.110755920410156, 32.800567626953125, 98.95478820800781, 189.04217529296875, 51.67023849487305, -34.687679290771484, 10.571609497070312, -13.546823501586914, 4.327531337738037, 37.100311279296875, -52.568572998046875, 35.18745422363281, -18.277851104736328, 36.62827682495117, 58.38801956176758, 35.13035583496094, 136.73402404785156, 25.822784423828125, 64.62828063964844, 85.48301696777344, 71.1038818359375, 84.22352600097656, 35.42414093017578, 67.65845489501953, -24.136030197143555, 11.538043975830078, 43.7457389831543, 88.38336944580078, 3.632418155670166, 121.17706298828125, 38.21854782104492, -54.0447998046875, 100.61064147949219, 104.09684753417969, 66.33949279785156, 14.225399017333984, 21.959653854370117, 27.785615921020508, 85.97517395019531, 40.518375396728516, -57.00838851928711, -2.874398708343506, 34.815128326416016, -31.093595504760742, 14.039507865905762, 113.43789672851562, 64.25528717041016, 35.84355926513672, 78.95516204833984, 35.32097244262695, 87.2996597290039, 68.86042785644531, -15.785491943359375, 160.6723175048828, -28.549243927001953, 24.205249786376953, 23.654436111450195, -24.60041618347168, -19.859586715698242, 50.87514114379883, -40.59874725341797, -27.814010620117188, 41.25919723510742, 13.061400413513184, 35.166744232177734, 10.598833084106445, -13.40951919555664, 70.34864807128906, 44.68003845214844, 43.54359817504883, 88.37928009033203, -70.78775787353516, 66.31953430175781, 84.79345703125, 81.09296417236328, 110.59907531738281, 113.79766845703125, 44.966644287109375, 55.199462890625, -44.92258834838867, 59.910953521728516, -3.873598337173462, 43.32742691040039, -20.101106643676758, -38.94757843017578, 41.75043487548828, -14.513847351074219, 0.7871364951133728, -33.90449523925781, 53.42015075683594, 2.2893457412719727, 120.37091827392578, -53.86504364013672, 106.61359405517578, 29.765274047851562, 20.51869010925293, 5.677116394042969, 24.429410934448242, -8.61368465423584, 51.53335189819336, 60.868370056152344, -41.90797424316406, 7.123847961425781, 12.48643970489502, 10.210492134094238, 88.51056671142578, 0.8372958898544312, 81.66043090820312, 76.0491714477539, 52.397735595703125, 42.90461730957031, 109.92567443847656, 71.60530853271484, 89.3866195678711, -26.228878021240234, 21.775808334350586, -41.22005844116211, 82.76238250732422, 95.2501449584961, -3.30859375, 98.3108139038086, -15.271557807922363, 97.25881958007812, -1.7332926988601685, 72.66735076904297, 7.858851432800293, 1.9516241550445557, 106.46824645996094, 25.566808700561523, 118.02738189697266, -54.6224479675293, 37.762123107910156, -10.784692764282227, -46.512359619140625, -10.038798332214355, -2.8655612468719482, 52.9105224609375, -24.951011657714844, -18.19416618347168, -19.195070266723633, 131.44361877441406, 19.32503318786621, 66.8170394897461, 45.164520263671875, -2.9875099658966064, 88.81787872314453, -5.248854160308838, 71.6876449584961, 4.966373920440674, -15.439577102661133, 18.853759765625, 140.67506408691406, 67.80895233154297, 77.69218444824219, -20.10365104675293, 71.89228057861328, 91.21576690673828, -45.88412857055664, 77.12662506103516, -55.22230911254883, 52.68974304199219, 21.495689392089844, 19.064123153686523, -45.19223403930664, -9.665379524230957, 10.260319709777832, -28.628450393676758, -22.94561004638672, 22.15477180480957, 52.090728759765625, 40.62726974487305, -22.065412521362305, 31.190357208251953, -79.06149291992188, -19.856647491455078, 17.997249603271484, 59.4876708984375, 64.20926666259766, 7.081664085388184, -102.65703582763672, -42.945274353027344, 40.062652587890625, -0.00549490749835968, -80.45507049560547, 73.07938385009766, -30.705768585205078, 32.70549774169922, 11.724626541137695, 20.302684783935547, -0.5139036178588867, 8.467965126037598, -52.1581916809082, 18.80805015563965, 24.5858211517334, 1.7694752216339111, -77.90489196777344, 24.617074966430664, 13.552350044250488, -19.035160064697266, 72.81009674072266, -50.22457504272461, -25.22374725341797, -7.7010979652404785, -86.41477966308594, -5.128740310668945, -45.31031799316406, 68.85079956054688, 47.382545471191406, 76.61822509765625, -69.75602722167969, 51.114437103271484, 85.24811553955078, 3.3715336322784424, 59.65350341796875, 5.26365852355957, -99.53326416015625, 29.148609161376953, -7.636439323425293, 31.515926361083984, -51.9046745300293, -8.106818199157715, -11.630526542663574, 45.82563018798828, 47.9391975402832, 14.28601360321045, 138.77767944335938, -28.51651954650879, -0.26077333092689514, 33.25126647949219, -57.99507522583008, 37.12383270263672, 49.598487854003906, 28.545148849487305, 61.7857666015625, 3.97513484954834, 7.632770538330078, 44.331329345703125, -64.49808502197266, -23.191389083862305, 23.786983489990234, -7.307819843292236, 101.64810180664062, 42.01519775390625, 44.66708755493164, 6.272434234619141, 19.678388595581055, 33.973472595214844, 75.03456115722656, 13.018818855285645, -45.251041412353516, 94.5821304321289, 64.02247619628906, -26.783464431762695, 43.817012786865234, 73.12154388427734, -16.990633010864258, 5.885912895202637, 33.9691276550293, 24.886186599731445, -52.001304626464844, -29.378934860229492, -29.24021339416504, 53.20186233520508, -18.684139251708984, 103.24457550048828, -84.59603881835938, 158.6592559814453, 5.1002020835876465, 51.46479797363281, 19.272228240966797, 47.41742706298828, 5.701707363128662, 9.04554271697998, 1.7375264167785645, 36.83934783935547, 0.958901584148407, 2.1450984477996826, 14.770145416259766, -63.86921691894531, 84.12394714355469, 65.75274658203125, 28.565595626831055, 12.055743217468262, 17.132131576538086, -42.00177001953125, -14.654595375061035, 59.80424499511719, 30.580625534057617, 25.160478591918945, -1.8373315334320068, -29.83941078186035, 90.78434753417969, 60.48373031616211, 81.64192962646484, 62.906158447265625, 0.45970627665519714, -10.086334228515625, 28.436782836914062, 79.174560546875, 36.89946365356445, -10.729776382446289, 36.02606201171875, -39.73601150512695, 67.74202728271484, 36.134342193603516, 18.71845817565918, -43.045318603515625, 59.858856201171875, 30.539113998413086, 0.4786294996738434, 66.74464416503906, 20.932945251464844, 30.27683448791504, 24.433238983154297, -66.99858856201172, 70.75289154052734, -71.07329559326172, 31.06577491760254, 123.22715759277344, -1.731740951538086, 26.4781436920166, 76.56820678710938, -4.528936386108398, -18.144081115722656, -50.73824691772461, 40.83585739135742, 36.247642517089844, 52.153079986572266, -45.79242706298828, 68.93997192382812, 97.6160659790039, 53.1056022644043, 102.5225830078125, -27.68123435974121, -32.51438903808594, 47.305335998535156, -50.98131561279297, 36.90874099731445, 36.56149673461914, 58.43387985229492, 119.86166381835938, -25.13296890258789, 104.5382080078125, -57.80706787109375, 11.448225021362305, 0.6920590400695801, 113.35734558105469, 14.517548561096191, 42.99142074584961, 4.342622756958008, -46.86030960083008, -11.484989166259766, 78.59353637695312, 44.41571807861328, 115.52682495117188, 46.39225769042969, -45.79104232788086, -74.871826171875, 81.53250122070312, 35.43604278564453, -33.25453567504883, 54.489967346191406, -45.62889099121094, 3.8216373920440674, 31.13661766052246, 75.63336944580078, -14.860371589660645, 16.616933822631836, 69.46405029296875, 6.453986644744873, 30.089061737060547, -63.25455093383789, 30.09748077392578, 1.6938402652740479, 59.232391357421875, 95.20922088623047, 35.87504959106445, 24.492599487304688, -66.29783630371094, -6.928175449371338, 13.223556518554688, -25.733577728271484, 4.631651401519775, 15.774585723876953, 137.84283447265625, 41.30230712890625, 75.17865753173828, -57.326324462890625, 53.574119567871094, 35.1403923034668, 59.59101867675781, 5.227567195892334, 14.379380226135254, 92.87561798095703, -42.745338439941406, -37.8534049987793, 6.169450283050537, -10.071199417114258, -25.97723388671875, 3.2809853553771973, 119.22978973388672, 73.05684661865234, 18.843046188354492, 98.69817352294922, -33.966827392578125, -66.42652893066406, 12.907678604125977, -28.372224807739258, 30.64896583557129, -39.399967193603516, 10.537215232849121, -30.122140884399414, 32.50989532470703, -50.25648880004883, 85.25700378417969, 25.51726531982422, -50.640357971191406, 90.36540222167969, 14.05788803100586, -64.71646881103516, 70.98703002929688, -34.643592834472656, -21.287052154541016, 34.762245178222656, 10.30102825164795, 32.71182632446289, 15.020135879516602, -29.764190673828125, 44.52000045776367, 39.60307312011719, 123.04244995117188, 33.020999908447266, -15.524638175964355, -27.672801971435547, -86.80731201171875, 21.91566276550293, 44.928871154785156, -47.44209671020508, -47.57601547241211, 79.36444854736328, -56.63250732421875, 39.43734359741211, -45.90650939941406, 3.685744285583496, -15.080583572387695, 110.3964614868164, 26.01566505432129, 0.02000127173960209, -53.22929382324219, -1.8543974161148071, -54.277992248535156, 9.511600494384766, 66.43904876708984, 48.13328552246094, 29.015953063964844, -67.02777099609375, -37.72920608520508, -25.41105842590332, -5.1766743659973145, 111.1107406616211, -60.59608459472656, 79.35309600830078, 81.62621307373047, 31.38876724243164, 35.58671569824219, -14.42014217376709, -61.92322540283203, -3.752349853515625, 38.28130340576172, 33.323673248291016, 22.599021911621094, -89.44300842285156, 70.43925476074219, 86.28548431396484, -28.326326370239258, 24.134992599487305, -50.95577621459961, 138.90328979492188, -37.04750442504883, 88.53426361083984, -10.352753639221191, 121.0114517211914, 45.536354064941406, 72.28569793701172, 38.978843688964844, -133.10719299316406, 13.310635566711426, 41.75376510620117, 11.09160327911377, -12.890472412109375, -47.36420440673828, -8.59508991241455, 25.733640670776367, 33.89057159423828, 9.221840858459473, 64.12096405029297, 42.13399124145508, -23.95281982421875, 26.385705947875977, 32.40454864501953, 76.82913208007812, 0.3307468295097351, 24.208646774291992, 104.45496368408203, -7.775572776794434, -10.317363739013672, -0.5790125727653503, 46.0384635925293, 50.20332717895508, 27.899248123168945, -56.37470626831055, -45.35466766357422, 43.7254524230957, 78.21358489990234, 26.62551498413086, 46.28194046020508, -4.867629528045654, 39.476078033447266, 36.03895568847656, -16.533010482788086, 78.3086166381836, 42.59596252441406, 35.376583099365234, -53.361534118652344, 41.712242126464844, 44.57686996459961, 8.153833389282227, 36.35742950439453, 19.54088592529297, 102.29273986816406, -12.707195281982422, 28.913593292236328, -5.780320644378662, 31.184429168701172, 31.668100357055664, 76.92339324951172, 7.521857261657715, 12.42945384979248, -2.3773794174194336, -31.82093048095703, -56.59273147583008, 8.748922348022461, 7.968662738800049, 31.314132690429688, -0.43858376145362854, 76.040771484375, 6.991036415100098, 52.40524673461914, 15.664525032043457, -32.896785736083984, 98.68714904785156, 30.583080291748047, -19.089954376220703, -93.51769256591797, 77.5599594116211, 78.10962677001953, 38.92144012451172, 34.39606857299805, -15.948688507080078, 80.02281951904297, -95.39973449707031, 78.2538070678711, 34.287269592285156, 71.44503784179688, 38.39186477661133, -29.3037052154541, 39.04244613647461, 31.384939193725586, -61.52511978149414, -36.343528747558594, 32.67857360839844, 20.238683700561523, 69.26457977294922, -45.35675811767578, -38.36664581298828, 20.113685607910156, 45.27488327026367, -41.8370361328125, 27.842357635498047, -47.84389114379883, 104.71715545654297, 47.04371643066406, 28.426925659179688, -90.7085189819336, 74.43839263916016, 66.51337432861328, 83.13758850097656, 57.19498825073242, -75.39098358154297, 1.8927897214889526, 91.3959732055664, -54.998008728027344, 19.585664749145508, 58.82803726196289, 33.10786819458008, 44.43970489501953, -9.596270561218262, 80.00201416015625, 30.9982967376709, 9.232632637023926, -38.464111328125, 44.97828674316406, 7.748196125030518, -15.372650146484375, 59.528846740722656, 41.110599517822266, 40.140445709228516, 8.204314231872559, 29.242441177368164, 5.067495346069336, 97.88138580322266, -2.7234668731689453, 50.481956481933594, 25.26030921936035, -2.246032476425171, 54.65752029418945, -13.960298538208008, -57.2552490234375, 3.28912615776062, -50.929073333740234, -34.018707275390625, 13.674171447753906, 1.3918869495391846, 39.577415466308594, 32.34801483154297, -7.723773956298828, 2.654378890991211, -39.343605041503906, -27.323680877685547, -2.4895827770233154, 5.427933216094971, -15.196487426757812, 44.43922805786133, 50.95154571533203, -64.24773406982422, -1.2059448957443237, -43.45001983642578, -32.27863693237305, -57.32813262939453, 27.20470428466797, 16.102848052978516, 70.1497802734375, 74.60047149658203, 6.461197376251221, 57.228782653808594, -40.02204513549805, -155.29379272460938, 3.8832507133483887, 133.0308380126953, 9.841470718383789, 22.6431941986084, 59.154273986816406, 35.59565734863281, -45.81425857543945, 23.048686981201172, 44.55841827392578, -36.652557373046875, -25.129207611083984, -38.72764587402344, 58.066383361816406, -28.99854850769043, 94.05632019042969, -30.987714767456055, -64.69178771972656, -6.540106773376465, 17.33526611328125, 26.531333923339844, 49.73455047607422, 17.16228485107422, 92.15646362304688, 4.3423638343811035, -45.499820709228516, 55.29039764404297, 77.31693267822266, 5.935142993927002, -97.20051574707031, -61.563480377197266, 25.944149017333984, 32.26628112792969, -35.795223236083984, -1.1438195705413818, 10.5862398147583, -34.618717193603516, 50.84104537963867, 110.35575866699219, -48.944034576416016, 30.603113174438477, -72.37409210205078, -38.256385803222656, -33.107295989990234, 42.804447174072266, -10.159285545349121, -10.539902687072754, 35.906368255615234, -10.425521850585938, -27.397550582885742, -86.70889282226562, -24.20784568786621, -1.5481752157211304, 15.112131118774414, 109.23121643066406, 4.619904518127441, 19.128429412841797, 148.6097412109375, -49.350093841552734, -45.91659927368164, 0.555979311466217, 13.928899765014648, 75.20207977294922, 55.5334587097168, 30.695350646972656, 10.742733001708984, -38.92592239379883, -40.1118278503418, -6.9509782791137695, 56.20314407348633, -41.298736572265625, 82.32687377929688, 60.40531921386719, 16.877656936645508, 65.36714935302734, 7.286301612854004, -54.98371124267578, 28.28822135925293, 10.83298397064209, 131.10906982421875, 5.009696960449219, 73.72338104248047, 52.72743225097656, 104.00060272216797, 134.1405792236328, -49.38900375366211, 4.200197219848633, 15.538948059082031, -8.506389617919922, 66.04285430908203, 79.06018829345703, -83.83940887451172, -33.89889907836914, -3.4404025077819824, 88.52777099609375, 96.5325698852539, -63.916961669921875, -26.504167556762695, 61.931270599365234, 17.192983627319336, 61.49390411376953, -34.733619689941406, 1.2280532121658325, -41.59663391113281, -18.54278564453125, 53.40031814575195, -32.169063568115234, 14.805493354797363, -30.086463928222656, -29.42951774597168, 48.97570037841797, 7.371340751647949, 24.644161224365234, 4.848045825958252, -61.11338806152344, -35.89624786376953, -22.007165908813477, -38.789249420166016, 28.89910316467285, 83.98686218261719, 15.94465160369873, -34.779029846191406, 18.678525924682617, -24.102853775024414, -34.544403076171875, 74.10302734375, 84.80445098876953, 125.20227813720703, 123.16170501708984, 99.6589584350586, -7.296947002410889, 94.32085418701172, -30.01111602783203, 3.2156689167022705, -46.103660583496094, 55.509124755859375, -106.12071228027344, 31.930435180664062, 43.15626907348633, -11.771540641784668, 61.16304397583008, 24.146677017211914, -31.206003189086914, 41.19219207763672, 54.338199615478516, 58.986419677734375, 36.032230377197266, 18.29363250732422, -0.7189698815345764, 22.70014762878418, 54.1579704284668, 18.74468231201172, -15.060816764831543, 35.89438247680664, -3.5967981815338135, 78.69181060791016, -70.80514526367188, -5.957059383392334, -29.206573486328125, -11.014291763305664, 51.714664459228516, 78.13998413085938, 0.24818451702594757, -43.80841827392578, 98.39553833007812, -101.04547882080078, 4.211548805236816, -12.978373527526855, 7.433084487915039, 102.48865509033203, 61.569515228271484, -31.41183853149414, -28.501344680786133, -4.201896667480469, 33.22825622558594, 15.328022003173828, 50.79572677612305, -28.025724411010742, 57.17347717285156, -37.61811828613281, 89.9642562866211, 29.791141510009766, 42.02109909057617, 30.07037925720215, 87.5601577758789, -34.66207504272461, 33.502296447753906, 55.04844665527344, 77.19914245605469, 77.65064239501953, 8.574078559875488, 9.491547584533691, -65.28778076171875, -24.802398681640625, 15.40480899810791, 92.92724609375, -9.548710823059082, 40.11787414550781, -98.91840362548828, 0.3420857787132263, 8.887456893920898, 103.01106262207031, 69.27299499511719, 68.42668151855469, -18.287893295288086, 90.31277465820312, 12.024649620056152, -46.807228088378906, 96.46866607666016, 34.903297424316406, 34.26940155029297, -49.82767868041992, 79.572265625, -0.2418610006570816, 4.2777628898620605, 25.126123428344727, 62.57295608520508, 5.413939952850342, -18.284547805786133, 8.579999923706055, 16.538637161254883, -66.9355239868164, 20.365468978881836, 45.38024139404297, 66.10643768310547, 7.534481048583984, -27.971214294433594, -9.12673282623291, 64.59136199951172, 48.595977783203125, 58.166015625, 48.0538444519043, -27.84837532043457, -1.6990203857421875, 25.85340118408203, 96.95063781738281, 13.832897186279297, -31.179468154907227, 96.01561737060547, 50.810264587402344, -12.067675590515137, 49.865447998046875, -27.76759147644043, -36.84050750732422, -22.598526000976562, 14.827534675598145, 111.36314392089844, -24.215486526489258, -41.159366607666016, -10.761697769165039, 64.9017562866211, 81.31742858886719, 146.0056915283203, 50.586063385009766, 41.01566696166992, -4.517657279968262, 10.17463207244873, 26.97669792175293, -61.20650863647461, 35.494728088378906, 76.46101379394531, -64.710693359375, -10.287124633789062, 116.66320037841797, -44.58539581298828, -34.86982345581055, 90.11210632324219, 62.63874053955078, 13.362144470214844, 116.572998046875, -0.6210862994194031, -40.11935043334961, -32.79167556762695, -16.377939224243164, 39.3892822265625, 26.617441177368164, -26.3834228515625, -57.06318664550781, 25.964941024780273, 118.96087646484375, -79.58995056152344, 51.03223419189453, -28.873220443725586, 0.021072296425700188, 82.96173095703125, 31.68360137939453, -60.316741943359375, 25.769107818603516, 64.61504364013672, -10.046707153320312, -38.32622528076172, 18.384273529052734, 40.462486267089844, 53.14520263671875, 82.42598724365234, -55.53343963623047, 60.051116943359375, 26.546689987182617, -44.75921630859375, 49.37725067138672, -54.330135345458984, -4.4652791023254395, -16.4586238861084, 3.168625593185425, 14.87928295135498, 1.4296040534973145, 113.15239715576172, 44.40686798095703, 40.12935256958008, 70.65292358398438, 28.951515197753906, 34.33325958251953, 60.13484573364258, 27.491392135620117, 79.78572082519531, 38.074607849121094, 99.39019775390625, 37.09225845336914, 57.58058166503906, -47.02880859375, 30.97149658203125, 88.9068374633789, -10.534015655517578, 92.04582977294922, 26.610858917236328, 22.183700561523438, 76.2408447265625, -1.4004143476486206, 63.23690414428711, 62.588294982910156, 39.734375, 82.45384216308594, 52.176517486572266, 45.64106369018555, -33.35267639160156, 42.75829315185547, 48.24253463745117, 64.1058120727539, 13.575839042663574, 66.37796020507812, -15.128376960754395, 12.086640357971191, 71.52693176269531, 47.48749542236328, 38.866844177246094, -30.192672729492188, 18.865053176879883, -114.30046844482422, -2.8270010948181152, -67.43123626708984, 34.375, 26.493227005004883, 18.36992073059082, 20.72588348388672, 60.67957305908203, -10.911441802978516, 29.2159366607666, 87.10082244873047, -19.79338264465332, 37.707244873046875, 41.71607208251953, -33.358558654785156, 75.8993911743164, 61.61392593383789, 29.29047966003418, 104.79308319091797, -34.258914947509766, 36.71965026855469, 63.58400344848633, 104.4455337524414, -28.98587989807129, 33.723350524902344, 33.64067840576172, -14.42283821105957, 22.899635314941406, 7.625734806060791, 11.319517135620117, -21.751771926879883, 29.715452194213867, 28.038837432861328, -23.467992782592773, 70.64053344726562, 38.198760986328125, 81.30740356445312, 69.3338394165039, 90.76206970214844, 77.82617950439453, -46.084617614746094, 30.67837905883789, 60.16414260864258, -16.592918395996094, 122.51139068603516, 40.59458541870117, 69.46416473388672, 40.49076843261719, 32.183345794677734, 14.726875305175781, 12.852289199829102, 20.36020278930664, 74.46693420410156, 30.58376121520996, 82.5763168334961, -15.817729949951172, -5.6187543869018555, -7.325552463531494, -23.033796310424805, -64.77513122558594, 10.717612266540527, 55.77203369140625, 52.68091583251953, 59.506439208984375, 12.977232933044434, 26.24473762512207, 18.45855140686035, 57.16608428955078, 18.86587142944336, 32.37614059448242, 23.810768127441406, 5.527227878570557, 23.437471389770508, 114.03571319580078, -15.891399383544922, 22.883787155151367, 51.454097747802734, 32.992698669433594, 8.369121551513672, -60.585941314697266, 38.657318115234375, 24.922771453857422, -44.50212478637695, 74.4063491821289, -16.59368133544922, -1.9915080070495605, -76.79513549804688, 25.257463455200195, 68.72550964355469, 17.784482955932617, 22.902652740478516, 57.732574462890625, -25.32618522644043, 47.84716796875, 120.13024139404297, 81.43292236328125, -14.103346824645996, -2.7382776737213135, 45.92670440673828, 6.239892959594727, -71.39140319824219, 30.090749740600586, 11.080636978149414, 1.7795618772506714, 4.798404693603516, 5.280523300170898, 53.67728805541992, 39.25941467285156, -50.829627990722656, -5.223485469818115, 9.440125465393066, -10.738810539245605, -66.88388061523438, -47.63248062133789, 28.634944915771484, -40.93391036987305, 33.75059509277344, 0.026095464825630188, 56.70741271972656, -20.41950225830078, -8.331650733947754, 54.38551712036133, -31.996234893798828, 6.6669182777404785, -12.956636428833008, 66.87068176269531, -38.42013931274414, 12.787676811218262, -51.48164749145508, -36.06242370605469, 98.07701110839844, -51.825965881347656, 40.86399841308594, 126.49728393554688, 52.724334716796875, 18.37696647644043, 83.03726959228516, -62.64215850830078, -1.6843111515045166, 0.32516804337501526, 21.66064453125, 135.56658935546875, -2.0113775730133057, 44.80715560913086, 52.8978157043457, 74.05827331542969, 77.63623809814453, -78.443603515625, 140.11508178710938, -57.96592712402344, -19.596637725830078, -10.825115203857422, -47.03903579711914, 8.3123197555542, 50.12334442138672, 4.20627498626709, -15.624038696289062, 79.720458984375, -97.41822052001953, 26.086685180664062, 36.32085037231445, -27.666881561279297, 31.217084884643555, -46.95654296875, 68.72705078125, -6.548271179199219, 43.77542495727539, 29.55412483215332, 13.927781105041504, -42.61249542236328, 73.10958099365234, -26.740001678466797, 14.444637298583984, 13.8275728225708, -31.88568687438965, -52.10986328125, -23.405494689941406, 3.5217738151550293, 25.26247787475586, -8.354212760925293, -36.782859802246094, 2.1606109142303467, 80.16679382324219, -7.040411472320557, 33.95074462890625, 60.22172546386719, 3.0803022384643555, 59.234893798828125, 20.70742416381836, -2.7476203441619873, -62.01588821411133, -42.08795928955078, 74.37226104736328, 58.36658477783203, -37.15890884399414, 83.56537628173828, 29.76711654663086, -13.775649070739746, 12.956253051757812, 95.93834686279297, 73.84687805175781, 15.026726722717285, 135.174072265625, -56.28205871582031, -6.275757789611816, 2.816312313079834, 25.83262825012207, -30.834470748901367, 62.84758377075195, 61.23464584350586, -57.602508544921875, -35.098899841308594, 66.28829193115234, 33.78420639038086, 30.73822784423828, 87.92211151123047, 50.500911712646484, 10.626042366027832, -3.7038705348968506, 14.688704490661621, -15.8399019241333, 0.24033120274543762, -8.188840866088867, 54.3364372253418, -6.984185218811035, 121.63206481933594, 6.832244396209717, 72.98070526123047, 115.72882080078125, 80.4059829711914, -70.8819808959961, -15.004680633544922, -8.080499649047852, 14.10198974609375, 82.15676879882812, 3.0797524452209473, 28.01496124267578, 61.484825134277344, -25.585779190063477, -17.450328826904297, 30.763242721557617, -11.949018478393555, -75.73346710205078, -66.25514221191406, -14.912481307983398, 29.445556640625, 11.626644134521484, 37.293792724609375, 111.75917053222656, -21.5660457611084, 25.06257438659668, 60.11017608642578, -101.20121765136719, 119.3619155883789, 35.14908218383789, 35.0804443359375, 4.26654052734375, 4.346012592315674, 57.23448944091797, 14.331465721130371, -34.4426155090332, -47.9590950012207, -79.28785705566406, 38.03423309326172, 21.205686569213867, 49.23085021972656, 90.2537841796875, 47.90964126586914, 55.01223373413086, -23.801422119140625, 52.97903060913086, 85.78511810302734, 13.838484764099121, 50.23729705810547, 34.28721618652344, 50.356754302978516, 89.22746276855469, 12.387916564941406, -36.649330139160156, 109.64385986328125, -23.962236404418945, 119.83789825439453, 30.029314041137695, -30.103090286254883, 89.62256622314453, 6.7481465339660645, -26.85422706604004, 41.013465881347656, 17.62128448486328, -25.49456214904785, 50.362091064453125, -16.625030517578125, -9.422207832336426, -106.7068099975586, -29.865697860717773, -25.111148834228516, 19.08104133605957, 56.3769645690918, 30.253522872924805, -28.486907958984375, 31.82864761352539, 123.85274505615234, 47.476104736328125, -10.867945671081543, 88.63923645019531, 14.172934532165527, -0.28685691952705383, -50.797874450683594, -14.796594619750977, 74.79125213623047, 75.9530029296875, 31.867740631103516, 25.227615356445312, -13.656976699829102, 19.75988006591797, 51.5072135925293, -23.944442749023438, 82.983642578125, 9.371803283691406, 40.191558837890625, -34.60184860229492, 27.852508544921875, 65.61058044433594, -8.312058448791504, -28.833511352539062, -60.155006408691406, -9.35850715637207, 7.552672386169434, 27.11874008178711, 49.38699722290039, 70.52622985839844, -5.298966884613037, -9.750962257385254, -11.434545516967773, -3.077021598815918, 50.30784606933594, -16.895736694335938, -41.80445098876953, 11.223987579345703, 29.20792007446289, 0.6869423389434814, 22.645240783691406, -20.579387664794922, 26.615032196044922, 130.2498779296875, 41.35322952270508, 58.279842376708984, -59.04483413696289, 9.700172424316406, -2.17862868309021, 3.553438663482666, 61.9642219543457, -31.04521942138672, 69.0678482055664, 39.19907760620117, 9.138867378234863, -1.7428981065750122, 83.33811950683594, 95.89942169189453, -35.65855407714844, -13.86406135559082, 0.5904983282089233, -57.161895751953125, -54.34376907348633, 32.194549560546875, -17.17784309387207, 33.59427261352539, 63.47913360595703, 36.20195388793945, 39.34339141845703, 3.6682047843933105, 3.766507148742676, -16.20046615600586, -70.3458480834961, 25.690357208251953, 2.56105899810791, 69.76575469970703, 87.52632141113281, -7.798244953155518, 54.517555236816406, 2.8117198944091797, 1.8588907718658447, -40.487274169921875, 48.30364227294922, 97.54386138916016, -51.375308990478516, 67.89274597167969, 1.641133189201355, -18.314027786254883, -11.041945457458496, -61.44416427612305, -16.164278030395508, 47.17677688598633, -111.54801940917969, -35.126285552978516, 31.121707916259766, 115.72288513183594, 96.85981750488281, 52.70100021362305, 10.39966869354248, -1.342644453048706, 24.415367126464844, 38.473148345947266, 0.9074404239654541, 69.63727569580078, 78.46031188964844, 30.966806411743164, 56.2972297668457, -19.301883697509766, 51.314125061035156, 59.274505615234375, 76.49655151367188, 32.0118293762207, -19.83770179748535, 166.53428649902344, 77.60530853271484, 11.864638328552246, 81.32893371582031, -40.2021598815918, 16.129249572753906, -30.174226760864258, -35.608909606933594, 50.49382781982422, 102.79193878173828, 59.634891510009766, 65.93306732177734, -3.146557092666626, -23.899822235107422, -64.3306655883789, -48.212100982666016, 67.02558898925781, 39.70765686035156, 60.156768798828125, 65.7022705078125, 50.906715393066406, -10.091565132141113, 1.6511945724487305, 62.01216506958008, 43.22838592529297, -24.842697143554688, -2.158670425415039, 45.6608772277832, 28.385135650634766, 72.82949829101562, -19.043399810791016, 59.71953582763672, 30.749053955078125, 43.448394775390625, 3.0816943645477295, -12.465097427368164, 86.91507720947266, 24.549104690551758, 43.56785583496094, -49.45598602294922, 3.1178133487701416, 69.29518127441406, -6.889047145843506, 82.7925033569336, 6.255172252655029, 27.38692283630371, 62.18754959106445, 83.79248809814453, -14.404410362243652, -28.629375457763672, -32.30191421508789, 19.253665924072266, -99.18783569335938, 7.901388168334961, 21.274999618530273, 171.03395080566406, -0.5552910566329956, 33.87167739868164, 48.37535858154297, 82.14197540283203, 40.23207092285156, 44.88978958129883, 12.7185640335083, 98.7634506225586, 58.49553680419922, -87.55683135986328, -28.204668045043945, 42.73155975341797, 44.727970123291016, 5.831762790679932, 17.36933135986328, 110.87214660644531, 16.19905662536621, 25.657943725585938, -72.20646667480469, 61.174259185791016, 51.75632095336914, 93.42514038085938, 40.75514221191406, -29.663738250732422, 22.809343338012695, -34.11233139038086, -27.69864273071289, 73.60807037353516, -63.76996612548828, 33.87599182128906, -37.158538818359375, -3.415616512298584, 30.647216796875, -35.37019348144531, 84.57274627685547, -20.83070945739746, -15.222990989685059, -7.541775226593018, 6.610225200653076, -32.19166946411133, 12.755075454711914, -45.56431579589844, -61.68320083618164, 68.99340057373047, 26.063325881958008, -56.934844970703125, -31.63259506225586, 2.241856336593628, 25.049816131591797, 51.18166732788086, 43.22962951660156, 12.091652870178223, 14.145971298217773, 17.879924774169922, 92.30207824707031, 28.443307876586914, -8.169562339782715, 96.7886962890625, 37.62894821166992, 56.175350189208984, 45.414459228515625, 55.3324089050293, 116.36869049072266, 114.64485168457031, 117.89482879638672, -59.10285568237305, 141.3988800048828, -85.3675537109375, 160.62806701660156, -62.438026428222656, 41.749351501464844, -46.483436584472656, 13.43089485168457, 60.768165588378906, 86.12777709960938, -37.767547607421875, 2.4528138637542725, 49.99905014038086, 56.69986343383789, 38.149356842041016, -39.699275970458984, 0.7561705112457275, 55.039772033691406, -9.993949890136719, 47.03018569946289, 69.67211151123047, -109.54672241210938, 57.940521240234375, 65.29248046875, 14.755831718444824, -14.558804512023926, -7.54782247543335, 32.66552734375, 132.81626892089844, -19.43121910095215, -8.68989086151123, 17.626827239990234, -3.7977325916290283, 62.13602066040039, -19.060100555419922, 38.61701965332031, -15.902016639709473, 3.682189702987671, 8.490962982177734, -22.035417556762695, 45.275413513183594, -12.566109657287598, -20.453813552856445, 2.7195756435394287, 47.63465881347656, 23.71350860595703, 6.241612911224365, 23.969581604003906, 82.98513793945312, -29.93429183959961, -15.132319450378418, 3.285878896713257, 82.62675476074219, -0.9325394034385681, -40.03091812133789, -28.548664093017578, 72.83321380615234, 27.67792510986328, -17.681861877441406, 26.703779220581055, 4.272362232208252, 9.300671577453613, -2.022782802581787, -41.32284164428711, 14.392807960510254, 2.93131685256958, 51.96147155761719, -21.261051177978516, -70.45507049560547, -1.286207675933838, 76.47887420654297, 68.84306335449219, -61.56581497192383, -31.70847511291504, 81.4682388305664, 18.013580322265625, 0.006802573334425688, 21.760496139526367, 17.54730796813965, 35.13818359375, 3.475346565246582, 100.23445892333984, 36.02241897583008, 83.6008071899414, 57.44709777832031, -8.048184394836426, 75.16459655761719, -17.70400047302246, 55.970829010009766, 3.010180711746216, 0.6604720950126648, 6.6568803787231445, 7.593291282653809, -3.851026773452759, -98.09429168701172, 56.67949676513672, -44.49317932128906, -24.274972915649414, -149.17410278320312, 136.41455078125, 8.48732852935791, -65.9031982421875, -142.9000244140625, 5.615660190582275, -39.37427520751953, 57.249019622802734, 18.367652893066406, -12.047142028808594, 47.36619186401367, -11.113441467285156, -67.78976440429688, 63.966922760009766, -34.473472595214844, 7.658177852630615, 79.37406158447266, -40.861854553222656, 42.13127517700195, 45.97956466674805, 22.26096534729004, 135.148681640625, 17.750207901000977, 82.56197357177734, -24.52342414855957, 44.01637649536133, 78.3363265991211, 29.707046508789062, 77.22334289550781, 1.1155083179473877, 34.10685729980469, -39.21998977661133, 21.05018424987793, 35.42382049560547, -0.11137703061103821, 17.66352653503418, 31.31060791015625, -58.197608947753906, -27.333606719970703, -33.42444610595703, -38.277984619140625, 28.211162567138672, 2.5769217014312744, 0.702112078666687, 20.339584350585938, -96.05223083496094, 120.02455139160156, -17.916839599609375, 49.994956970214844, -32.89118957519531, -55.796817779541016, -31.03321647644043, 156.31141662597656, 15.848057746887207, 31.93499183654785, 24.460002899169922, -45.62983322143555, -21.17197608947754, 24.795940399169922, 101.46736907958984, -21.739362716674805, 60.045654296875, 65.00353240966797, 71.67236328125, 66.82731628417969, 32.5164680480957, -28.875944137573242, -118.54952239990234, -54.48407745361328, -45.66767883300781, 28.995731353759766, -20.850391387939453, -50.378814697265625, 83.64119720458984, 73.88504028320312, 44.48807144165039, 13.942643165588379, -6.009097099304199, 18.722797393798828, -49.5213737487793, 142.9711151123047, 6.515483856201172, -94.41494750976562, -78.08625030517578, 21.63457679748535, 41.11033248901367, 30.678253173828125, 75.56504821777344, 1.2737460136413574, 22.39618492126465, 104.26921844482422, -19.436555862426758, -30.43733024597168, -13.772584915161133, -38.11684799194336, -39.28756332397461, 13.894197463989258, -88.83648681640625, 26.921728134155273, -26.31784439086914, -13.352018356323242, 80.0302734375, -3.715036153793335, 46.99671936035156, 5.395283222198486, -32.421443939208984, 30.006332397460938, -6.952003479003906, 30.574113845825195, -60.55092239379883, -39.74897766113281, 31.60430145263672, 97.10588073730469, 111.57742309570312, -6.158607482910156, 34.93779754638672, -49.209110260009766, -11.258709907531738, 23.389616012573242, 28.754314422607422, 102.63530731201172, 4.1498308181762695, -11.104349136352539, -52.7349739074707, 9.715289115905762, 53.62689208984375, 69.02690124511719, -24.49764633178711, 85.69589233398438, 87.6298828125, 34.5471076965332, 39.8986930847168, -12.736190795898438, -60.348758697509766, -43.77656173706055, 112.02388000488281, 57.65369415283203, 22.2794189453125, 43.57537078857422, 89.74663543701172, 17.16853141784668, 34.69248962402344, 60.938167572021484, -43.545188903808594, 7.41130256652832, 71.57428741455078, 58.3846435546875, -47.48073196411133, 23.15130615234375, 150.7803497314453, 76.15779876708984, 1.2106850147247314, 34.70609664916992, -22.165294647216797, -24.085208892822266, -20.37521743774414, 47.74919128417969, -60.33291244506836, 98.90644836425781, -38.37611770629883, 67.0182113647461, 64.79740142822266, 57.2032470703125, 5.171209812164307, -13.672086715698242, 20.468626022338867, 174.09071350097656, 49.30792999267578, 60.29801559448242, -42.1652717590332, 56.7772331237793, 13.313663482666016, 10.858210563659668, 39.18297576904297, 40.237369537353516, -17.869054794311523, 3.823906183242798, -0.0875808596611023, -6.560810565948486, 50.259918212890625, 68.3290786743164, 62.55009841918945, 5.306661605834961, 88.62460327148438, 12.348109245300293, -12.658517837524414, 40.17540740966797, -31.19702911376953, 19.71302032470703, 67.36146545410156, -16.4227294921875, -15.907502174377441, 3.592444658279419, 60.53666687011719, -9.798648834228516, -0.8619897365570068, -47.56886291503906, 67.8698959350586, 47.89685821533203, -23.401203155517578, -57.128902435302734, -12.702549934387207, 29.607080459594727, 43.710777282714844, 68.57223510742188, -10.017797470092773, 85.73715209960938, -43.68745803833008, 36.475528717041016, 37.91987991333008, 77.0180435180664, 12.315567016601562, 23.87510871887207, -12.429295539855957, 26.19131088256836, -9.179534912109375, 13.479992866516113, 31.559791564941406, 39.15702438354492, 75.71507263183594, 15.659899711608887, 32.633018493652344, -26.376907348632812, 34.31479263305664, 52.82422637939453, -31.96155548095703, -51.70491409301758, 70.0728530883789, 1.6938109397888184, 41.6087760925293, 107.94011688232422, 102.78311157226562, 15.979994773864746, -22.18277931213379, 38.98930740356445, -71.81985473632812, 66.437744140625, 30.80518913269043, 32.19343185424805, 50.35422897338867, -61.80345153808594, -83.47109985351562, 67.47364807128906, 47.59480285644531, 132.068115234375, 18.746994018554688, 114.49385833740234, 1.6340264081954956, 61.150760650634766, -14.80085277557373, 46.24101638793945, -0.950869083404541, 33.12762451171875, 92.93280029296875, -8.570898056030273, 26.117956161499023, -27.291851043701172, 48.792423248291016, 38.36466979980469, 55.95267868041992, 89.27056884765625, 97.48310852050781, -39.3489990234375, 135.1729736328125, 56.29823684692383, 37.83168411254883, 13.621537208557129, -39.4095458984375, 9.779600143432617, -68.30955505371094, -15.771896362304688, -33.54474639892578, -33.25424575805664, -48.74039840698242, 21.1395206451416, -0.9795171022415161, 3.481191635131836, -15.045207977294922, 37.333343505859375, 93.6551742553711, 50.10686492919922, 87.54248046875, 12.337952613830566, 31.80742835998535, 21.859106063842773, -13.064740180969238, -13.79413890838623, 55.465946197509766, -6.908392906188965, -16.047954559326172, -51.2508430480957, 137.17137145996094, -75.4010009765625, -39.45151138305664, 90.95698547363281, 57.16976547241211, 48.61481857299805, -45.793697357177734, -0.13845586776733398, 20.67930793762207, 19.266145706176758, -32.81602478027344, 0.31546521186828613, 26.275728225708008, 45.64158248901367, 69.08869171142578, 35.82708740234375, 2.1748404502868652, 175.06805419921875, 63.55827713012695, 72.55558013916016, -46.06739044189453, -47.6383056640625, 22.757341384887695, 9.547606468200684, -32.53676986694336, 39.265499114990234, 76.00426483154297, 77.4614486694336, 36.975807189941406, -37.966278076171875, 49.53924560546875, -25.408891677856445, 39.97130584716797, -52.12179183959961, 51.048858642578125, 6.902658939361572, 96.26085662841797, -24.3114070892334, 126.46910095214844, 55.19888687133789, 19.167917251586914, -0.22884565591812134, 27.272602081298828, 95.97380065917969, -49.780372619628906, 63.24204635620117, 64.54542541503906, 22.856672286987305, -23.107542037963867, 43.60818862915039, -3.7978785037994385, -20.240501403808594, -4.787469863891602, 75.40392303466797, 12.529577255249023, 19.954448699951172, 14.79517650604248, 2.789710760116577, -54.66727828979492, -24.594655990600586, 62.33271789550781, -39.952850341796875, 78.46731567382812, 29.92885398864746, 32.43011474609375, 46.67415237426758, 110.02041625976562, 72.52070617675781, 32.56142044067383, 48.009613037109375, 8.792881965637207, -47.92243194580078, -50.85648727416992, -3.7922322750091553, -12.677654266357422, -10.3795166015625, 49.826942443847656, -56.91905975341797, -26.06780433654785, -7.147940158843994, -61.54735565185547, -39.13246536254883, -14.941193580627441, -1.6151714324951172, -24.260557174682617, 46.637569427490234, -51.13002395629883, -93.15567779541016, 61.80027389526367, -71.318115234375, 93.05843353271484, 15.371126174926758, -41.51654815673828, 70.48020935058594, -25.11467742919922, 45.65447998046875, -59.03886795043945, 111.464599609375, 42.78325271606445, 24.58075523376465, 48.3213996887207, 75.51738739013672, -55.11088180541992, 16.772571563720703, -12.436559677124023, 0.38882943987846375, 10.831323623657227, -14.596915245056152, 69.49675750732422, 52.33787536621094, 1.93538236618042, 111.84324645996094, 3.457568883895874, -63.48288345336914, 77.85014343261719, 39.009395599365234, 32.451988220214844, 17.106706619262695, 93.63094329833984, -101.97985076904297, -88.52041625976562, 20.15353012084961, 24.758026123046875, 17.494417190551758, -35.17365646362305, 3.296262741088867, 72.63982391357422, -14.786857604980469, 47.22286605834961, -26.126781463623047, 43.72919845581055, -66.49424743652344, 33.739234924316406, 57.75284957885742, 2.7320339679718018, -52.44725036621094, -15.584399223327637, 1.010195255279541, -13.03719425201416, -66.67151641845703, 4.790798187255859, -35.637020111083984, -9.490459442138672, 10.63609790802002, 4.285010814666748, -60.94806671142578, -13.271927833557129, 25.061573028564453, 49.981285095214844, 59.50586700439453, 21.39128875732422, -148.41909790039062, -24.718582153320312, 64.5152816772461, 2.137688159942627, 3.122805118560791, -5.345864295959473, -26.777936935424805, 36.12750244140625, 81.50057983398438, 11.146129608154297, 56.2979850769043, 130.12493896484375, 50.94165802001953, -3.782494068145752, -3.67386794090271, -42.03315734863281, -24.3267822265625, 65.07996368408203, 134.14791870117188, 13.766162872314453, 39.723567962646484, -30.004016876220703, -26.977291107177734, 61.33956527709961, -29.528173446655273, 5.949120998382568, 68.37557983398438, 97.0606689453125, 24.445205688476562, 32.9168586730957, 14.452919006347656, 0.9655941128730774, 55.614620208740234, 56.11068344116211, -22.56746482849121, 10.078020095825195, 22.51661491394043, 76.5587387084961, -21.933612823486328, 49.168975830078125, 105.45708465576172, -16.18624496459961, 36.649566650390625, -10.154524803161621, 68.37306213378906, -8.833429336547852, 18.26809310913086, 31.89339828491211, -37.3051643371582, 13.089632987976074, 62.939735412597656, -29.531352996826172, 134.43577575683594, 66.52476501464844, 29.275829315185547, 98.17699432373047, 53.546669006347656, 64.2716293334961, 144.09754943847656, -90.3451156616211, -4.144075870513916, 66.79045104980469, -82.15145874023438, 62.08128356933594, 16.786794662475586, -6.606523513793945, 80.5263671875, 44.50185775756836, 47.02914810180664, -25.424030303955078, -38.25271224975586, -5.1723198890686035, 138.0692596435547, 20.510501861572266, 47.5166130065918, 53.05224609375, 10.931260108947754, 128.0680389404297, 14.767532348632812, -2.0019097328186035, -24.859359741210938, 62.98151779174805, 43.68656921386719, 3.72851300239563, 85.75625610351562, 98.77450561523438, 12.111106872558594, -5.208629608154297, 8.068710327148438, 70.78742980957031, 13.402933120727539, 76.47129821777344, -22.262975692749023, 78.65108489990234, -95.66020965576172, -26.47855567932129, 54.87053298950195, 52.83053970336914, -0.013430073857307434, 2.3823866844177246, -43.50349426269531, -3.244218587875366, 68.56646728515625, -12.332831382751465, -13.646759986877441, 17.916383743286133, 37.153141021728516, 48.71867752075195, 68.60568237304688, -17.24466323852539, -54.024208068847656, 41.91933822631836, 104.67206573486328, 13.49455738067627, -59.72798156738281, -5.612226963043213, 11.66901683807373, -9.421680450439453, 89.5085220336914, -17.133108139038086, 62.201385498046875, 85.99707794189453, -27.536270141601562, -57.60722351074219, -48.18564224243164, 36.37759017944336, 96.56929016113281, 33.09886169433594, 94.87078857421875, -41.002532958984375, 9.257545471191406, 61.350345611572266, 27.640262603759766, 38.844947814941406, 14.200410842895508, -15.241061210632324, 9.404837608337402, 110.52819061279297, -85.54185485839844, -14.367606163024902, 30.146455764770508, -0.6656703352928162, 78.45831298828125, 91.26750946044922, 7.564453125, -41.61730194091797, 9.294637680053711, 51.2483024597168, 13.330431938171387, 1.27281653881073, -52.52458190917969, 143.21884155273438, 26.291040420532227, 38.24831771850586, -38.167240142822266, 49.608524322509766, -61.73976516723633, -19.709007263183594, 43.999610900878906, -89.47793579101562, -31.10683250427246, 15.301025390625, -19.060895919799805, -28.130542755126953, 53.17658615112305, 40.89348602294922, 64.62501525878906, -19.2351016998291, 50.47001647949219, -11.208642959594727, 56.001224517822266, -39.59571075439453, 27.84379768371582, -53.74668884277344, 18.095705032348633, 46.0007209777832, 96.39419555664062, 118.05070495605469, 69.42223358154297, 54.447410583496094, 88.64913177490234, -96.69864654541016, 83.22684478759766, -0.517480194568634, 118.21190643310547, 74.66707611083984, -32.06943893432617, -12.975065231323242, -41.046321868896484, -35.405147552490234, 6.135080814361572, 53.035133361816406, 7.0087504386901855, -57.569557189941406, 69.88990783691406, -25.546424865722656, 19.97705841064453, 11.889341354370117, 15.165143013000488, 52.575923919677734, -13.457548141479492, 92.21527862548828, 60.093772888183594, 12.847818374633789, 94.70796966552734, 1.0335153341293335, 61.614559173583984, 35.85124969482422, 23.105485916137695, 93.35014343261719, 101.12162780761719, 32.89012145996094, 36.189388275146484, -32.02291488647461, 100.08316040039062, 39.12535858154297, 22.627483367919922, -4.479650020599365, 6.1138105392456055, -10.792390823364258, 93.35576629638672, 13.237730979919434, 12.65381145477295, 57.473426818847656, 30.806270599365234, -32.21413803100586, 80.77346801757812, 18.152935028076172, 81.7125015258789, 13.594338417053223, -10.61772346496582, -58.508216857910156, 57.75394058227539, 40.346981048583984, 11.087101936340332, 0.06434570997953415, 53.82558059692383, 42.747493743896484, 16.680330276489258, -64.4664306640625, -17.873111724853516, 36.05427169799805, 6.768372535705566, 33.91029739379883, 101.28949737548828, 29.029199600219727, 60.83546829223633, 26.142818450927734, 57.62084197998047, 82.6343002319336, -5.1988844871521, -10.232397079467773, -26.357255935668945, 149.8009796142578, 8.164924621582031, 32.863182067871094, 93.18991088867188, 23.747371673583984, 18.791912078857422, 24.587160110473633, 27.64673614501953, 61.59690856933594, 62.84402847290039, 30.179044723510742, 68.61485290527344, 123.6149673461914, 19.325115203857422, -40.905181884765625, -98.28992462158203, 36.405296325683594, -14.294564247131348, -7.613813877105713, 0.0679139494895935, 4.757599830627441, 76.9781723022461, -12.250197410583496, -19.608863830566406, -58.82496643066406, 83.89800262451172, 58.15249252319336, 49.67626190185547, -19.734861373901367, 28.525367736816406, 35.81196975708008, 70.75731658935547, 19.933815002441406, 55.34941864013672, 88.65782928466797, 78.84452056884766, 37.93051528930664, 23.254653930664062, -25.316181182861328, 66.91319274902344, -25.31106185913086, 23.551410675048828, -14.521313667297363, -45.945404052734375, 23.989788055419922, -26.339412689208984, -0.07646282017230988, 81.10285186767578, 49.74057388305664, 11.083559036254883, 96.94132995605469, 27.026981353759766, 68.3760986328125, 66.692138671875, 92.01760864257812, -38.77219009399414, 68.20440673828125, 47.00600051879883, 4.388326168060303, 28.485525131225586, -48.042057037353516, 114.40994262695312, 2.133362054824829, 104.24681091308594, 58.733943939208984, -7.081042289733887, -74.79850006103516, 42.705772399902344, 77.17045593261719, -56.62164306640625, -40.846588134765625, -18.600921630859375, 85.57427978515625, 26.080198287963867, -83.1091079711914, -70.22904968261719, -52.5950813293457, -53.67877197265625, 0.8607825040817261, 0.17544962465763092, 9.283968925476074, 85.36185455322266, -18.320274353027344, 2.0455360412597656, -0.2606934607028961, 13.879499435424805, -19.137210845947266, 57.24895095825195, 12.472016334533691, 50.55997848510742, -56.374732971191406, -9.893892288208008, 3.3750288486480713, -16.803098678588867, 103.294921875, 100.4593505859375, 70.37962341308594, 30.88772201538086, 14.847660064697266, 53.24236297607422, 9.544078826904297, -95.6737289428711, -40.21204376220703, -31.101654052734375, -23.77262306213379, 40.83685302734375, 9.2518949508667, 5.96867561340332, 82.88278198242188, 91.55038452148438, 50.650028228759766, 56.53374099731445, 54.00018310546875, 55.94607162475586, -21.77059555053711, 56.32529067993164, -30.535078048706055, 84.64114379882812, 49.28697204589844, 55.29086685180664, 43.70315170288086, -0.8928133249282837, 61.21044921875, 42.36700439453125, -0.47983649373054504, -13.08945369720459, -19.376670837402344, 90.65655517578125, -5.062204837799072, -31.453067779541016, 81.2074203491211, 50.254146575927734, 15.797281265258789, -27.74723243713379, 88.92694854736328, 35.15306091308594, -26.76569366455078, 19.796178817749023, -18.509410858154297, -6.093733310699463, 49.17854690551758, 20.401466369628906, 13.651296615600586, -8.96005630493164, 66.6673812866211, 35.9912223815918, 31.90400505065918, 32.384578704833984, 40.965511322021484, 73.609375, -31.242088317871094, -61.27972412109375, 32.64744186401367, 27.586410522460938, -56.67338562011719, 61.9293212890625, 92.19265747070312, 71.37799835205078, 37.58545684814453, -36.07457733154297, 89.3375244140625, 104.7723388671875, 28.118253707885742, 89.61127471923828, -30.061899185180664, 33.79218292236328, -18.168041229248047, -30.39751625061035, -9.301225662231445, 48.348106384277344, 104.38617706298828, -44.64189529418945, -32.79092788696289, 89.48957824707031, 121.40631866455078, 61.51097106933594, 58.473846435546875, 58.92523193359375, -13.924772262573242, 66.56183624267578, -37.89363098144531, 33.549312591552734, 26.522979736328125, -20.641042709350586, 4.411351203918457, -1.9257603883743286, 67.53343200683594, -0.4071919918060303, 30.33285903930664, 47.20360565185547, 68.95329284667969, 18.86735725402832, 64.17730712890625, 62.13177490234375, 22.406368255615234, -1.932202696800232, 26.6685848236084, 16.217350006103516, 1.70356023311615, 27.426725387573242, -29.022716522216797, 80.38722229003906, 60.776058197021484, 59.75495529174805, -55.25563049316406, -7.766024112701416, 90.35018920898438, 27.217992782592773, -113.7251205444336, 37.510276794433594, -62.05702209472656, 64.5312271118164, 0.6831771731376648, -40.18522262573242, -36.534324645996094, 32.278995513916016, -22.58709716796875, -34.92920684814453, 43.62117385864258, 62.91061019897461, 20.388927459716797, -13.50760269165039, 34.6864128112793, 39.64872741699219, 123.94064331054688, 45.702327728271484, 36.43466567993164, 79.25968170166016, 28.000202178955078, -43.73988342285156, 60.58211135864258, -20.72492027282715, 61.8425407409668, -108.79805755615234, 81.42650604248047, -13.997476577758789, -43.054351806640625, 24.51907730102539, 40.59117126464844, -1.0950347185134888, 10.359369277954102, -13.944511413574219, 87.82080078125, 22.10968017578125, 22.94108772277832, 8.368026733398438, 44.5494270324707, 53.47888946533203, -44.137935638427734, 28.854434967041016, 23.917213439941406, 38.766910552978516, -97.89971923828125, -55.817718505859375, 34.98807907104492, 46.13377380371094, -4.05185604095459, -26.933042526245117, -78.1155014038086, 92.8034896850586, 55.4304313659668, -16.35042381286621, 59.85737609863281, 0.4583803713321686, 23.492862701416016, -10.697515487670898, -18.33523178100586, 22.008150100708008, -40.03622817993164, 14.493609428405762, -48.92045593261719, -28.654294967651367, 17.068973541259766, 51.54069137573242, 81.20816040039062, 80.53548431396484, 94.44109344482422, 60.14249801635742, 80.9974136352539, 15.580689430236816, 28.868480682373047, -81.86034393310547, 72.82107543945312, 54.81534957885742, 29.88397216796875, -10.519771575927734, -55.25161361694336, -74.99463653564453, 5.3565568923950195, 7.106452941894531, -12.993252754211426, 35.24374771118164, 20.875572204589844, -13.052056312561035, -45.560630798339844, 81.44055938720703, -131.63644409179688, 27.497657775878906, 94.59423065185547, 53.78876876831055, 32.60385513305664, 53.617652893066406, 55.43863296508789, -86.94603729248047, -22.93353843688965, 117.07463073730469, 8.471394538879395, -36.04384231567383, 96.42129516601562, 5.262608528137207, -16.43267250061035, 71.56963348388672, 124.13899230957031, 3.1963376998901367, 71.09638214111328, 81.6823959350586, 50.85055923461914, 64.17396545410156, -19.304561614990234, 17.13282585144043, -33.7056999206543, 43.331260681152344, -19.17826271057129, 60.99781799316406, -19.97728157043457, -23.708784103393555, 79.84131622314453, 58.32181167602539, 29.699975967407227, -44.33420181274414, 0.8275234699249268, 84.64523315429688, 131.46958923339844, 87.30776977539062, 39.79823684692383, 0.5804643034934998, 43.11808395385742, 18.40935707092285, 67.91181182861328, -25.72101593017578, 45.806724548339844, 9.326849937438965, 32.933712005615234, 108.3792953491211, 9.827629089355469, 49.912200927734375, -18.234577178955078, 51.60823440551758, 34.56608200073242, -19.252731323242188, 31.89876937866211, -33.864986419677734, 88.10435485839844, -30.819766998291016, -64.66265106201172, -2.759565830230713, 35.70984649658203, -64.60002899169922, -17.522106170654297, -42.437774658203125, -55.01156234741211, 68.26639556884766, 16.06243896484375, 43.37925720214844, 53.138431549072266, -8.130149841308594, 19.635189056396484, 19.332338333129883, -4.0365705490112305, 55.259395599365234, -19.010801315307617, -8.114347457885742, 50.48644256591797, -0.31382259726524353, 56.107147216796875, 70.06478118896484, 21.653696060180664, 65.34657287597656, 47.261714935302734, -80.16671752929688, -30.83484649658203, 22.52515983581543, 125.21765899658203, 119.6497573852539, 19.15493392944336, 42.345706939697266, -11.763607025146484, 98.80535125732422, 20.289791107177734, -36.16474151611328, -31.106964111328125, -4.937602519989014, 57.9046516418457, -4.811717510223389, 61.93552017211914, 26.58340835571289, -39.95631790161133, -1.42864990234375, 16.1933650970459, 40.18717575073242, 81.52459716796875, 77.15345764160156, -9.310150146484375, 47.64021682739258, -0.9569714069366455, -67.49544525146484, 76.61185455322266, 101.06171417236328, -13.358960151672363, -60.176048278808594, 96.04293823242188, 120.52915954589844, 9.98241901397705, 29.725139617919922, 34.76652908325195, -102.05512237548828, 14.683032989501953, 78.64962768554688, -92.40296173095703, 100.15546417236328, 10.402117729187012, -9.572460174560547, 21.61949920654297, 22.555011749267578, -26.633968353271484, -13.818377494812012, 76.78038024902344, 71.66878509521484, 24.703325271606445, 46.766441345214844, -42.37754440307617, -51.067291259765625, 11.696788787841797, -37.47517776489258, -7.250168323516846, 20.805971145629883, 14.235235214233398, 81.60509490966797, -27.852731704711914, 32.91547393798828, 142.77317810058594, -34.93134307861328, 35.73147201538086, 6.665077209472656, 60.694271087646484, 47.9044303894043, 16.96100616455078, -11.018315315246582, -19.57707405090332, 75.96953582763672, -4.270779609680176, -6.503625869750977, 70.69631958007812, 22.59638786315918, 34.19954299926758, 12.932963371276855, -69.07601928710938, 25.45294189453125, 14.461291313171387, 67.09893798828125, -45.37284851074219, -10.530631065368652, 131.41847229003906, -67.6173095703125, 40.841407775878906, 39.045841217041016, 122.94403839111328, 37.39545440673828, 80.48779296875, -31.44696617126465, 31.330284118652344, -11.951568603515625, -11.459757804870605, 58.324256896972656, 17.33718490600586, 4.241857528686523, -16.933063507080078, 67.66397857666016, 60.43280029296875, 46.43800354003906, -24.059480667114258, 49.04722213745117, 83.85356140136719, 1.026167869567871, -31.500802993774414, 27.22170066833496, 25.593952178955078, -62.650596618652344, 166.82041931152344, 52.61347579956055, 26.907949447631836, -3.64482045173645, 63.064125061035156, -9.617178916931152, -48.107940673828125, 13.735701560974121, 14.191351890563965, 17.49437141418457, -24.895906448364258, -107.85481262207031, 97.81822204589844, 37.129390716552734, 12.450194358825684, 101.55328369140625, 122.83728790283203, 34.30152893066406, 87.0184097290039, 38.792816162109375, 56.47053909301758, -7.5171942710876465, 57.00154495239258, 54.83491516113281, -5.096970558166504, 9.073817253112793, -25.76959228515625, 60.218841552734375, 97.87023162841797, -7.207147598266602, -40.97092056274414, -36.49428176879883, -18.43537712097168, 62.60139465332031, -67.09585571289062, -15.319061279296875, 27.877920150756836, 102.28023529052734, 31.144947052001953, -3.6935060024261475, -29.94534683227539, 96.01103973388672, -13.881119728088379, 65.46266174316406, -63.04537582397461, -33.37074661254883, 32.648094177246094, -17.41986656188965, -1.2974754571914673, -29.729963302612305, 108.23150634765625, -39.78848648071289, 6.271788120269775, 81.9225082397461, 38.85635757446289, 16.690738677978516, 74.41773223876953, -23.676496505737305, 18.644689559936523, 72.14579010009766, 96.00770568847656, -24.735254287719727, 36.69918441772461, 82.37281036376953, -68.8785171508789, 156.89280700683594, -90.55237579345703, 63.96067810058594, 27.505491256713867, 5.707364559173584, 66.33889770507812, -19.388568878173828, 22.587554931640625, -0.7489133477210999, 100.4649887084961, 12.313323020935059, 23.630640029907227, -37.74627685546875, 43.55116653442383, -35.14436721801758, 6.110895156860352, -27.318880081176758, 69.78643035888672, -18.879911422729492, -35.67424392700195, 14.881203651428223, 11.01634693145752, 36.665374755859375, -14.47058391571045, 32.054744720458984, 105.76089477539062, 80.48975372314453, -13.845902442932129, -6.9665207862854, 66.75806427001953, 25.5902042388916, 53.03696823120117, -52.60793685913086, -42.473018646240234, 133.69204711914062, 40.00187301635742, 14.443169593811035, 7.604189872741699, 93.06340789794922, -85.59455108642578, -61.31980514526367, -24.667560577392578, -36.7596549987793, -26.151071548461914, -2.4129483699798584, 51.73611831665039, 108.85353088378906, 82.2040023803711, -78.97685241699219, -14.205965995788574, 15.145782470703125, 46.474735260009766, 52.83064651489258, 63.51518249511719, 87.2808609008789, -45.076263427734375, 35.5603141784668, 90.8768081665039, 60.141815185546875, -19.27065086364746, 101.1951904296875, 15.451960563659668, -72.44261169433594, 16.89913558959961, 39.429466247558594, -8.65162181854248, -76.36412811279297, -78.12973022460938, 26.06678581237793, 16.662057876586914, -69.18754577636719, -8.307586669921875, 6.053776264190674, 36.7741813659668, 84.9823226928711, -4.07022762298584, -34.03900146484375, -19.76396369934082, 2.78104567527771, 31.444334030151367, -23.883394241333008, 1.0541011095046997, 17.3959903717041, -13.917112350463867, 29.647624969482422, -49.339534759521484, -46.2906494140625, 18.46561050415039, -39.85594177246094, 15.000221252441406, 14.741509437561035, 22.512704849243164, 30.13758659362793, -5.14105224609375, -39.201759338378906, -15.981755256652832, 3.6573433876037598, 33.866661071777344, -66.82974243164062, 50.125526428222656, 86.45071411132812, 95.9918212890625, 57.03575134277344, -12.417373657226562, 92.89196014404297, 80.59382629394531, -56.084232330322266, 6.626631259918213, -60.46497344970703, 26.88658905029297, 57.40225601196289, 10.65061092376709, 55.73124313354492, 34.45949935913086, 26.89019012451172, -2.7523276805877686, -8.855774879455566, -112.36773681640625, 38.041324615478516], "dtype": "float32"}], "e": -155.29379272460938}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(array_min, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")